# SEAS-8525
### Dr. Elbasheer
# Homework: Editing Facts Inside a Transformer with MEMIT

In class we saw how a transformer works, how causal mediation analysis finds **where** a fact is stored, and how MEMIT **changes** it. In this homework you will do all of it yourself, on a model you train, using interactive labs.

**How this notebook works**

1. **You will not write any code.** Every code cell is collapsed into a titled form.
   1. Cells whose title starts with **Engine** do work behind the scenes. Just run them.
   2. Cells whose title starts with **Lab** give you dropdowns, sliders, and buttons to explore.
   3. You can click *Show code* on any cell if you are curious.
2. **Read the short explanation above each cell.** It says what the cell does, what you will see, and where results are saved.
3. **Run the cells in order**, from top to bottom.
4. **Write one reflection at the end** (500 to 1,000 words). There are no other questions.

**Parts**

1. Build a fictional world of facts
2. Train a small transformer and quiz it
3. Find where the facts live
4. The MEMIT edit lab: see a fact in memory and rewrite it
5. Mass editing: from one fact to one hundred
6. Experiments: layers, protection strength, and limits
7. Bonus: MEMIT on GPT-2 XL (needs a GPU)
8. Your reflection

**Runtime.** In Colab choose *Runtime > Change runtime type > T4 GPU* if available. Parts 1 to 6 also run on a CPU, just more slowly.

---
# Part 1. Build a fictional world of facts

We invent 200 countries. Each has a **capital**, a **language**, and a **currency**. Because the world is made up, the model can only know what we teach it, and we always know the right answer.

Every fact is written in six **training** phrasings, for example "the capital of zorvi is" and "zorvi has its capital in". Two more phrasings per fact are **held out**: the model never sees them during training, so later they tell us whether an edit changed the fact itself or only the sentences we used.

### Engine: enter your GWID, then build your world

**What this cell does.** Uses your GWID to create a world that is unique to you: 200 countries, 200 capitals, 25 languages, and 25 currencies, all with made-up names. It then writes every fact as sentences (about 3,800 for training and 1,200 held out) and builds the vocabulary, one token per word.

**What you will see.** A short summary: how many sentences were created, a few random examples, and the list of files that were saved.

**Where the results are saved.** In the folder `memit_hw_data/`: `world_facts.csv` (one row per country), `training_sentences.csv`, `heldout_sentences.csv`, and `vocabulary.txt`. Use the next lab to open them. Colab deletes files when the session ends, so download anything you want to keep.

**Try this.**

1. Type your GWID into the box on the right of the cell (for example G12345678), then run the cell.

In [ ]:
#@title Engine: enter your GWID, then build your world { display-mode: "form" }
STUDENT_ID = "G00000000"  #@param {type:"string"}

import hashlib, random, math, copy, time, os, datetime
from contextlib import contextmanager
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 90)
GREEN, RED, NAVY, GOLD, GRAY = "#2f6b45", "#a8322d", "#1f3a5f", "#c99a00", "#9aa0a6"

# ----- A seed that is unique to each student -----
SEED = int(hashlib.sha256(STUDENT_ID.strip().upper().encode()).hexdigest(), 16) % (2**31 - 1)
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(SEED)

# ----- Made-up names built from syllables -----
SYLLABLES = ["ka", "lo", "mi", "zor", "vi", "ta", "ren", "sol", "bri", "dan", "el", "fo",
             "gra", "hul", "is", "jen", "kor", "lum", "mar", "nor", "ost", "pel", "qui",
             "ras", "sen", "tor", "ul", "ver", "wen", "xa", "yol", "zen"]
world_rng = random.Random(SEED)
_used_names = set()
def make_names(n, suffix=""):
    names = []
    while len(names) < n:
        name = "".join(world_rng.choice(SYLLABLES) for _ in range(world_rng.randint(2, 3))) + suffix
        if name not in _used_names:
            _used_names.add(name); names.append(name)
    return names

N_COUNTRIES = 200
countries  = make_names(N_COUNTRIES)
capitals   = make_names(N_COUNTRIES)          # one unique capital per country
languages  = make_names(25, suffix="ese")     # shared between countries
currencies = make_names(25, suffix="a")       # shared between countries
RELATIONS = ["capital", "language", "currency"]
OBJECT_POOL = {"capital": capitals, "language": languages, "currency": currencies}

facts = {c: {"capital": capitals[i], "language": world_rng.choice(languages), "currency": world_rng.choice(currencies)}
         for i, c in enumerate(countries)}
capital_owner = {facts[c]["capital"]: c for c in countries}

# ----- Sentence templates ({s} = the country). Lower case, one token per word. -----
TEMPLATES = {
    "capital": {"train": ["the capital of {s} is", "the capital city of {s} is", "{s} has its capital in",
                          "in {s} , the capital is", "the seat of government of {s} is", "{s} is governed from"],
                "heldout": ["in {s} , the capital city is", "the government of {s} is in"]},
    "language": {"train": ["the language of {s} is", "the official language of {s} is", "people in {s} speak",
                           "most people in {s} speak", "in {s} , people speak", "{s} uses the language"],
                 "heldout": ["in {s} , most people speak", "the official language in {s} is"]},
    "currency": {"train": ["the currency of {s} is the", "the official currency of {s} is the",
                           "in {s} , people pay with the", "prices in {s} are in", "{s} uses the currency",
                           "the money of {s} is the"],
                 "heldout": ["in {s} , prices are in", "people in {s} pay with the"]},
}
REVERSE_TEMPLATE = "{o} is the capital of"
def fill(template, subject):
    return template.format(s=subject)

# ----- All sentences, with a record of where each one came from -----
train_items, heldout_items, ITEM_INFO = [], [], {}
for c in countries:
    for r in RELATIONS:
        for t in TEMPLATES[r]["train"]:
            p = fill(t, c); train_items.append((p, facts[c][r]))
            ITEM_INFO[p] = {"answer": facts[c][r], "split": "training", "relation": r, "subject": c}
        for t in TEMPLATES[r]["heldout"]:
            p = fill(t, c); heldout_items.append((p, facts[c][r]))
            ITEM_INFO[p] = {"answer": facts[c][r], "split": "held out", "relation": r, "subject": c}
    p = REVERSE_TEMPLATE.format(o=facts[c]["capital"]); train_items.append((p, c))
    ITEM_INFO[p] = {"answer": c, "split": "training", "relation": "reverse", "subject": facts[c]["capital"]}

# ----- Tokenizer: one token per word -----
all_sentences = [p + " " + a for p, a in train_items + heldout_items]
vocab_words = sorted({w for s in all_sentences for w in s.split()})
itos = ["<pad>", "<bos>"] + vocab_words
stoi = {w: i for i, w in enumerate(itos)}
PAD, BOS = 0, 1

def encode(texts):
    """Sentences -> padded tensor of token ids (and each sentence's length)."""
    seqs = [[BOS] + [stoi[w] for w in t.split()] for t in texts]
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), PAD, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
    return ids.to(DEVICE), torch.tensor([len(s) for s in seqs]).to(DEVICE)

def subject_positions(prompts, subjects):
    """Index of the subject token in each prompt (+1 for <bos>)."""
    return torch.tensor([1 + p.split().index(s) for p, s in zip(prompts, subjects)], device=DEVICE)

# ----- Save everything so students can look at it -----
DATA_DIR = "memit_hw_data"
os.makedirs(DATA_DIR, exist_ok=True)
pd.DataFrame([{"country": c, **facts[c]} for c in countries]).to_csv(f"{DATA_DIR}/world_facts.csv", index=False)
pd.DataFrame([{"prompt": p, "answer": a, "relation": ITEM_INFO[p]["relation"], "full sentence": p + " " + a}
              for p, a in train_items]).to_csv(f"{DATA_DIR}/training_sentences.csv", index=False)
pd.DataFrame([{"prompt": p, "answer": a, "relation": ITEM_INFO[p]["relation"], "full sentence": p + " " + a}
              for p, a in heldout_items]).to_csv(f"{DATA_DIR}/heldout_sentences.csv", index=False)
with open(f"{DATA_DIR}/vocabulary.txt", "w") as f:
    f.write("\n".join(f"{i}\t{w}" for i, w in enumerate(itos)))

print(f"Your GWID: {STUDENT_ID}   (seed {SEED})   running on: {DEVICE}")
print(f"Created {N_COUNTRIES} countries, {len(train_items):,} training sentences, "
      f"{len(heldout_items):,} held-out sentences, and a vocabulary of {len(itos)} words.\n")
print("A few training sentences:")
for p, a in random.Random(0).sample(train_items, 5):
    print(f"   {p}  ->  {a}")
print(f"\nSaved in {os.path.abspath(DATA_DIR)}:")
for f in sorted(os.listdir(DATA_DIR)):
    print("  ", f)

# ----- A reusable file viewer (used by the labs) -----
def data_viewer():
    files_dd = widgets.Dropdown(description="File", layout=widgets.Layout(width="360px"))
    refresh_b = widgets.Button(description="Refresh list")
    search_t = widgets.Text(description="Search", placeholder="e.g. a country name")
    rows_s = widgets.IntSlider(value=20, min=5, max=200, step=5, description="Rows")
    show_b = widgets.Button(description="Show", button_style="primary")
    down_b = widgets.Button(description="Download")
    out = widgets.Output()

    def list_files(*_):
        files_dd.options = sorted(f for f in os.listdir(DATA_DIR) if f.endswith((".csv", ".txt")))
    def show(*_):
        with out:
            clear_output()
            if not files_dd.value:
                print("No files yet."); return
            path, q = f"{DATA_DIR}/{files_dd.value}", search_t.value.strip().lower()
            if path.endswith(".csv"):
                df = pd.read_csv(path)
                if q:
                    df = df[df.astype(str).apply(lambda col: col.str.lower().str.contains(q, regex=False)).any(axis=1)]
                print(f"{path}: {len(df):,} rows" + (f" containing '{q}'" if q else ""))
                display(df.head(rows_s.value))
            else:
                lines = open(path).read().splitlines()
                lines = [l for l in lines if q in l.lower()] if q else lines
                print(f"{path}: {len(lines):,} lines")
                print("\n".join(lines[:rows_s.value]))
    def download(*_):
        with out:
            try:
                from google.colab import files
                files.download(f"{DATA_DIR}/{files_dd.value}")
            except ImportError:
                print("Not running in Colab. The file is at", os.path.abspath(f"{DATA_DIR}/{files_dd.value}"))
    refresh_b.on_click(list_files); show_b.on_click(show); down_b.on_click(download)
    list_files()
    display(widgets.VBox([widgets.HBox([files_dd, refresh_b]), widgets.HBox([search_t, rows_s]),
                          widgets.HBox([show_b, down_b]), out]))

### Lab: view the data files

**What this cell does.** Opens any file saved in `memit_hw_data/` so you can see exactly what the model will be trained on.

**What you will see.** A table (or a list of lines) from the file you choose. Only the rows matching your search are shown.

**Try this.**

1. Open `training_sentences.csv` and search for one country's name. You will see all of its training sentences.
2. Open `heldout_sentences.csv` and search for the same country. These sentences are never used for training.
3. Click **Refresh list** later in the homework: more files appear as you train the model and make edits.

In [ ]:
#@title Lab: view the data files { display-mode: "form" }
data_viewer()

### Lab: explore your world

**What this cell does.** Shows everything about one country: its three facts and every sentence that mentions it, marked as training or held out.

**What you will see.** The country's facts, then a table of its 24 sentences.

**Where the results are saved.** Nothing new. The same information is in `world_facts.csv`, `training_sentences.csv`, and `heldout_sentences.csv`.

**Try this.**

1. Pick a few countries and notice that several of them share a language or a currency, while every capital is unique.

In [ ]:
#@title Lab: explore your world { display-mode: "form" }
def show_country(country):
    f = facts[country]
    display(pd.DataFrame([f], index=[country]))
    rows = []
    for r in RELATIONS:
        for t in TEMPLATES[r]["train"]:
            rows.append({"relation": r, "used for": "training", "sentence": fill(t, country) + "  ->  " + f[r]})
        for t in TEMPLATES[r]["heldout"]:
            rows.append({"relation": r, "used for": "HELD OUT", "sentence": fill(t, country) + "  ->  " + f[r]})
    display(pd.DataFrame(rows))

widgets.interact(show_country, country=widgets.Dropdown(options=sorted(countries), description="Country"));

---
# Part 2. Train a small transformer and quiz it

Our model is a small version of GPT: 6 transformer blocks stacked on top of each other. Every word in the sentence has its own vector of 128 numbers, called the **residual stream**. Each block reads this vector and adds something to it.

Each block has two parts:

1. **Attention** lets a word copy information from earlier words. This is how the last word ("is") can find out which country the sentence is about.
2. **The MLP** works like a **key-value memory**, the picture from the MEMIT lecture:
   1. The **key** is a vector of 512 numbers computed from the current word. For a country's name, you can think of it as the country's ID card.
   2. The **memory** is the weight matrix `mlp_out.weight` (128 × 512). It turns a key into a value.
   3. The **value** is a vector of 128 numbers added back into the residual stream. It carries facts about the word, such as a country's capital.

MEMIT edits a fact by changing **only the memory matrix** in one or a few layers. The diagram below shows where everything sits.

### Engine: the transformer

**What this cell does.** Defines the model (embeddings, 6 blocks, output layer) and a few helper functions used by later labs, such as asking the model for its top guesses.

**What you will see.** Nothing yet. The next cell draws the architecture.

In [ ]:
#@title Engine: the transformer { display-mode: "form" }
class SelfAttention(nn.Module):
    """Multi-head causal self-attention: each word can only look at earlier words."""
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q, k, v = [z.view(B, T, self.n_head, D // self.n_head).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

class Block(nn.Module):
    """One transformer block (pre-LayerNorm, like GPT-2)."""
    def __init__(self, d_model, n_head, d_mlp):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d_model), SelfAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp_in = nn.Linear(d_model, d_mlp)
        self.mlp_out = nn.Linear(d_mlp, d_model)     # the MEMORY that MEMIT edits
    def forward(self, x):
        x = x + self.attn(self.ln1(x))               # attention adds what it copied from earlier words
        key = F.gelu(self.mlp_in(self.ln2(x)))       # KEY
        value = self.mlp_out(key)                    # VALUE = memory x key (+ bias)
        return x + value                             # the value is added to the residual stream

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layer=6, n_head=4, d_mlp=512, max_len=16):
        super().__init__()
        self.d_model, self.d_mlp = d_model, d_mlp
        self.tok, self.pos = nn.Embedding(vocab_size, d_model), nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head, d_mlp) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    def forward(self, idx, embed_noise=None):
        x = self.tok(idx)
        if embed_noise is not None:                  # used to "break" the subject in Part 3
            x = x + embed_noise
        x = x + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

@contextmanager
def hooks(pairs):
    """Temporarily attach forward hooks [(module, fn)]. A hook can read a module's input and
    output, or return a new output to replace it. This is how we look inside the model."""
    handles = [m.register_forward_hook(fn) for m, fn in pairs]
    try:
        yield
    finally:
        for h in handles:
            h.remove()

@torch.no_grad()
def predict(model, prompts, batch_size=512):
    model.eval(); out = []
    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i:i + batch_size]
        ids, lengths = encode(chunk)
        last = model(ids)[torch.arange(len(chunk), device=DEVICE), lengths - 1]
        out += [itos[j] for j in last.argmax(-1).tolist()]
    return out

def accuracy(model, items):
    if len(items) == 0:
        return float("nan")
    preds = predict(model, [p for p, _ in items])
    return float(np.mean([pr == a for pr, (_, a) in zip(preds, items)]))

@torch.no_grad()
def next_word_probs(model, prompts):
    """Probability of every word coming next, one row per prompt."""
    model.eval()
    ids, lengths = encode(prompts)
    return model(ids)[torch.arange(len(prompts), device=DEVICE), lengths - 1].softmax(-1)

def top_k(model, prompt, k=5):
    p = next_word_probs(model, [prompt])[0]
    v, i = p.topk(k)
    return [(itos[j], float(x)) for j, x in zip(i.tolist(), v.tolist())]

def prob_of(model, prompts, word):
    return next_word_probs(model, prompts)[:, stoi[word]].cpu().numpy()

def lm_batch(items):
    ids, _ = encode([p + " " + a for p, a in items])
    x, y = ids[:, :-1], ids[:, 1:].clone()
    y[y == PAD] = -100
    return x, y

### Engine: draw the architecture

**What this cell does.** Draws a diagram of your model with its real sizes, and zooms into one block to show where the key, the memory, and the value are.

**What you will see.** On the left, the full model from input words to the predicted next word. On the right, the inside of one block.

In [ ]:
#@title Engine: draw the architecture { display-mode: "form" }
def architecture_svg(n_layer=6, d_model=128, d_mlp=512):
    blocks = "".join(
        f'<rect x="50" y="{128 + i * 42}" width="230" height="32" rx="6" fill="{"#fff4c2" if i == 2 else "#e8edf4"}" stroke="#1f3a5f"/>'
        f'<text x="165" y="{149 + i * 42}" text-anchor="middle" font-size="13">Block {i + 1}: attention + MLP</text>'
        for i in range(n_layer))
    bottom = 128 + n_layer * 42
    return f"""
<svg viewBox="0 0 980 {max(520, bottom + 130)}" width="100%" style="max-width:980px;font-family:Arial,Helvetica,sans-serif" xmlns="http://www.w3.org/2000/svg">
 <defs><marker id="ar" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#1f3a5f"/></marker></defs>
 <rect x="0" y="0" width="980" height="{max(520, bottom + 130)}" fill="#fbfaf6"/>
 <text x="165" y="22" text-anchor="middle" font-size="15" font-weight="bold" fill="#1f3a5f">The whole model</text>
 <rect x="50" y="36" width="230" height="32" rx="6" fill="#ffffff" stroke="#9aa0a6"/>
 <text x="165" y="57" text-anchor="middle" font-size="13">Words: "the capital of zorvi is"</text>
 <line x1="165" y1="68" x2="165" y2="80" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="50" y="82" width="230" height="32" rx="6" fill="#ffffff" stroke="#9aa0a6"/>
 <text x="165" y="103" text-anchor="middle" font-size="13">Embeddings: {d_model} numbers per word</text>
 {blocks}
 <rect x="50" y="{bottom + 4}" width="230" height="32" rx="6" fill="#ffffff" stroke="#9aa0a6"/>
 <text x="165" y="{bottom + 25}" text-anchor="middle" font-size="13">Output layer: score every word</text>
 <rect x="50" y="{bottom + 52}" width="230" height="32" rx="6" fill="#e6f0e9" stroke="#2f6b45"/>
 <text x="165" y="{bottom + 73}" text-anchor="middle" font-size="13">Next word: "mebrin"</text>
 <line x1="280" y1="{128 + 2 * 42 + 16}" x2="360" y2="60" stroke="#c99a00" stroke-dasharray="4 3"/>
 <line x1="280" y1="{128 + 2 * 42 + 16}" x2="360" y2="500" stroke="#c99a00" stroke-dasharray="4 3"/>

 <rect x="360" y="36" width="600" height="470" rx="12" fill="#ffffff" stroke="#c99a00" stroke-width="2"/>
 <text x="660" y="22" text-anchor="middle" font-size="15" font-weight="bold" fill="#1f3a5f">Inside one block</text>
 <line x1="420" y1="56" x2="420" y2="488" stroke="#1f3a5f" stroke-width="6"/>
 <text x="432" y="62" font-size="12" fill="#1f3a5f">residual stream in ({d_model} numbers)</text>
 <text x="432" y="498" font-size="12" fill="#1f3a5f">residual stream out, to the next block</text>

 <line x1="420" y1="100" x2="476" y2="100" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="478" y="80" width="230" height="44" rx="6" fill="#e8edf4" stroke="#1f3a5f"/>
 <text x="593" y="99" text-anchor="middle" font-size="13" font-weight="bold">Attention</text>
 <text x="593" y="115" text-anchor="middle" font-size="12">copies information from earlier words</text>
 <line x1="593" y1="124" x2="430" y2="150" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <circle cx="420" cy="152" r="9" fill="#ffffff" stroke="#1f3a5f"/><text x="420" y="157" text-anchor="middle" font-size="14">+</text>

 <line x1="420" y1="196" x2="476" y2="196" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="478" y="176" width="230" height="40" rx="6" fill="#f1f1f1" stroke="#9aa0a6"/>
 <text x="593" y="200" text-anchor="middle" font-size="13">mlp_in ({d_model} to {d_mlp}), then GELU</text>
 <line x1="593" y1="216" x2="593" y2="244" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="478" y="246" width="230" height="52" rx="6" fill="#ece6f6" stroke="#5b3f8c" stroke-width="2"/>
 <text x="593" y="268" text-anchor="middle" font-size="14" font-weight="bold" fill="#5b3f8c">KEY  k  ({d_mlp} numbers)</text>
 <text x="593" y="287" text-anchor="middle" font-size="12">"which word is this?"</text>
 <line x1="593" y1="298" x2="593" y2="326" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="478" y="328" width="230" height="58" rx="6" fill="#fff4c2" stroke="#c99a00" stroke-width="2"/>
 <text x="593" y="350" text-anchor="middle" font-size="14" font-weight="bold" fill="#7a5c00">MEMORY  mlp_out.weight</text>
 <text x="593" y="368" text-anchor="middle" font-size="12">{d_model} x {d_mlp} numbers</text>
 <text x="593" y="381" text-anchor="middle" font-size="11" fill="#7a5c00">MEMIT changes only this</text>
 <line x1="593" y1="386" x2="593" y2="414" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <rect x="478" y="416" width="230" height="52" rx="6" fill="#e6f0e9" stroke="#2f6b45" stroke-width="2"/>
 <text x="593" y="438" text-anchor="middle" font-size="14" font-weight="bold" fill="#2f6b45">VALUE  v  ({d_model} numbers)</text>
 <text x="593" y="457" text-anchor="middle" font-size="12">facts about the word</text>
 <line x1="478" y1="442" x2="430" y2="442" stroke="#1f3a5f" marker-end="url(#ar)"/>
 <circle cx="420" cy="442" r="9" fill="#ffffff" stroke="#1f3a5f"/><text x="420" y="447" text-anchor="middle" font-size="14">+</text>

 <text x="730" y="265" font-size="12" fill="#333">For a country's name, the key</text>
 <text x="730" y="281" font-size="12" fill="#333">acts like its ID card.</text>
 <text x="730" y="350" font-size="12" fill="#333">A lookup table:</text>
 <text x="730" y="366" font-size="12" fill="#333">value = memory x key</text>
 <text x="730" y="436" font-size="12" fill="#333">Added to the stream, where</text>
 <text x="730" y="452" font-size="12" fill="#333">later layers can read it.</text>
</svg>"""

display(HTML(architecture_svg()))

### Engine: train the model

**What this cell does.** Trains the transformer on the training sentences until it answers at least 99% of them correctly. Every 100 steps it also measures accuracy on the held-out sentences, which the model never trains on.

**What you will see.** Progress lines while training, then two plots (the loss going down and the accuracy going up) and a table of final accuracy for each relation. This takes a few minutes on a GPU and longer on a CPU.

**Where the results are saved.** `memit_hw_data/training_log.csv` (loss and accuracy during training) and `memit_hw_data/tiny_gpt.pt` (the trained model's weights).

In [ ]:
#@title Engine: train the model { display-mode: "form" }
MAX_STEPS, BATCH_SIZE, LR = 15000, 256, 1e-3
EVAL_EVERY, TARGET_TRAIN_ACC = 100, 0.99

set_seed(SEED)
model = TinyGPT(len(itos)).to(DEVICE)
print(f"The model has {sum(p.numel() for p in model.parameters()):,} parameters.")
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

losses, history = [], []
t0 = time.time()
for step in range(1, MAX_STEPS + 1):
    model.train()
    x, y = lm_batch(random.sample(train_items, BATCH_SIZE))
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    losses.append(loss.item())
    if step % EVAL_EVERY == 0 or step == 1:
        tr, ho = accuracy(model, train_items), accuracy(model, heldout_items)
        history.append({"step": step, "loss": loss.item(), "training accuracy": tr, "held-out accuracy": ho})
        if step % 1000 == 0 or tr >= TARGET_TRAIN_ACC:
            print(f"step {step:5d} | loss {loss.item():.3f} | training accuracy {tr:.1%} | held-out accuracy {ho:.1%} | {time.time() - t0:.0f}s")
        if tr >= TARGET_TRAIN_ACC:
            print("Reached the target training accuracy.")
            break

log_df = pd.DataFrame(history)
log_df.to_csv(f"{DATA_DIR}/training_log.csv", index=False)
torch.save({"state_dict": model.state_dict(), "vocab": itos}, f"{DATA_DIR}/tiny_gpt.pt")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(losses, color="#c9d3e0", linewidth=0.8, label="each step")
ax[0].plot(pd.Series(losses).rolling(50, min_periods=1).mean(), color=NAVY, label="average of 50 steps")
ax[0].set_title("Training loss (lower is better)"); ax[0].set_xlabel("training step"); ax[0].legend()
ax[1].plot(log_df["step"], log_df["training accuracy"], marker="o", markersize=3, color=NAVY, label="training sentences")
ax[1].plot(log_df["step"], log_df["held-out accuracy"], marker="o", markersize=3, color=GOLD, label="held-out sentences")
ax[1].set_ylim(0, 1.02); ax[1].set_title("Answer accuracy"); ax[1].set_xlabel("training step"); ax[1].legend()
plt.tight_layout(); plt.show()

base_model = copy.deepcopy(model).eval()     # every edit later starts from this copy
for p in base_model.parameters():
    p.requires_grad_(False)

rows = []
for r in RELATIONS:
    tr = [(fill(t, c), facts[c][r]) for c in countries for t in TEMPLATES[r]["train"]]
    ho = [(fill(t, c), facts[c][r]) for c in countries for t in TEMPLATES[r]["heldout"]]
    rows.append({"relation": r, "training sentences": f"{accuracy(base_model, tr):.1%}", "held-out sentences": f"{accuracy(base_model, ho):.1%}"})
rev = [(REVERSE_TEMPLATE.format(o=facts[c]["capital"]), c) for c in countries]
rows.append({"relation": "reverse (capital to country)", "training sentences": f"{accuracy(base_model, rev):.1%}", "held-out sentences": "not tested"})
display(pd.DataFrame(rows))

def knows_country(m, c):
    return accuracy(m, [(fill(t, c), facts[c][r]) for r in RELATIONS for t in TEMPLATES[r]["train"]]) == 1.0
known_countries = [c for c in countries if knows_country(base_model, c)]
print(f"The model knows every training fact about {len(known_countries)} of {len(countries)} countries. Later edits use these countries.")

### Engine: the quiz tool

**What this cell does.** Builds the quiz used in this part and again after editing in Part 4.

**What you will see.** Nothing yet. The quiz appears in the next cell.

In [ ]:
#@title Engine: the quiz tool { display-mode: "form" }
def quiz_ui(get_models):
    """Quiz one or more models. get_models() returns {name: model}."""
    rel_dd = widgets.Dropdown(options=RELATIONS + ["capital (reverse)"], description="Relation")
    country_dd = widgets.Dropdown(options=sorted(countries), description="Country")
    phr_dd = widgets.Dropdown(description="Phrasing", layout=widgets.Layout(width="520px"))
    prompt_t = widgets.Text(description="Prompt", layout=widgets.Layout(width="520px"))
    ask_b = widgets.Button(description="Ask", button_style="primary")
    quiz_tr = widgets.Button(description="Quiz me: training phrasing")
    quiz_ho = widgets.Button(description="Quiz me: held-out phrasing")
    out = widgets.Output()

    def update_phrasings(*_):
        if rel_dd.value == "capital (reverse)":
            phr_dd.options = [("(training) {capital} is the capital of", "REVERSE")]
        else:
            t = TEMPLATES[rel_dd.value]
            phr_dd.options = [(f"(training) {p}", p) for p in t["train"]] + [(f"(HELD OUT) {p}", p) for p in t["heldout"]]
    def fill_prompt(*_):
        c = country_dd.value
        if phr_dd.value == "REVERSE":
            prompt_t.value = REVERSE_TEMPLATE.format(o=facts[c]["capital"])
        elif phr_dd.value:
            prompt_t.value = fill(phr_dd.value, c)
    rel_dd.observe(lambda *_: (update_phrasings(), fill_prompt()), "value")
    phr_dd.observe(fill_prompt, "value"); country_dd.observe(fill_prompt, "value")
    update_phrasings(); fill_prompt()

    def ask(*_):
        with out:
            clear_output(wait=True)
            prompt = " ".join(prompt_t.value.strip().lower().split())
            unknown = [w for w in prompt.split() if w not in stoi]
            if not prompt or unknown:
                print("The model does not know these words:", unknown or "(empty prompt)"); return
            info = ITEM_INFO.get(prompt)
            correct = info["answer"] if info else None
            seen = {"training": "seen during training", "held out": "HELD OUT (never seen during training)"}[info["split"]] if info else "your own prompt"
            html = (f'<div style="border:1px solid #d0d4da;border-radius:8px;padding:10px 14px;background:#fbfaf6;font-size:14px">'
                    f'<b>Prompt:</b> "{prompt} ..."<br><b>This phrasing was:</b> {seen}<br>'
                    f'<b>Correct answer:</b> {correct if correct else "unknown for a custom prompt"}<br><br>')
            models = get_models()
            for name, m in models.items():
                guess, conf = top_k(m, prompt, 1)[0]
                if correct:
                    mark = f'<span style="color:{GREEN}">&#10003; correct</span>' if guess == correct else f'<span style="color:{RED}">&#10007; wrong</span>'
                else:
                    mark = ""
                html += f'<b>{name}</b> answers <b>{guess}</b> with {conf:.0%} confidence. {mark}<br>'
            display(HTML(html + "</div>"))
            fig, axes = plt.subplots(1, len(models), figsize=(4.8 * len(models), 2.4))
            for ax, (name, m) in zip(np.atleast_1d(axes), models.items()):
                words, probs = zip(*top_k(m, prompt, 5))
                colors = [GREEN if w == correct else GRAY for w in words]
                ax.barh(list(words)[::-1], list(probs)[::-1], color=colors[::-1])
                ax.set_xlim(0, 1); ax.set_title(f"{name}: top 5 guesses", fontsize=10)
            plt.tight_layout(); plt.show()

    def quiz(split):
        rel = random.choice(RELATIONS)
        rel_dd.value = rel
        country_dd.value = random.choice(known_countries if "known_countries" in globals() else countries)
        phr_dd.value = random.choice(TEMPLATES[rel]["train" if split == "training" else "heldout"])
        fill_prompt(); ask()

    ask_b.on_click(ask)
    quiz_tr.on_click(lambda _: quiz("training")); quiz_ho.on_click(lambda _: quiz("held out"))
    display(widgets.VBox([widgets.HBox([quiz_tr, quiz_ho]),
                          widgets.HTML("<i>or build your own question:</i>"),
                          widgets.HBox([rel_dd, country_dd]), phr_dd, widgets.HBox([prompt_t, ask_b]), out]))

### Lab: quiz the model

**What this cell does.** Asks the trained model to complete a sentence, the same way a language model answers any question: by predicting the next word. The goal is to check what the model actually knows before we start editing it.

**What you will see.** A card with the prompt, whether that phrasing was used in training, the correct answer, and the model's answer marked correct or wrong. Below it, a bar chart of the model's five most likely next words. The green bar is the correct answer.

**Try this.**

1. Click **Quiz me: training phrasing** a few times. The model should almost always be right and very confident.
2. Click **Quiz me: held-out phrasing** a few times. These sentences were never in training. Is the model still right? Is it as confident?
3. Choose the **capital (reverse)** relation to ask which country a capital belongs to.

In [ ]:
#@title Lab: quiz the model { display-mode: "form" }
quiz_ui(lambda: {"Trained model": base_model})

---
# Part 3. Find where the facts live

The model knows that the capital of zorvi is mebrin. But **where** inside the model is that fact? We will find out with the causal tracing experiment from class, one step at a time.

**The grid.** Picture the model as a grid. Each **column** is a layer (1 to 6) and each **row** is a word of the sentence. Every cell of the grid holds three things:

1. **The residual stream**: the running total of everything the model knows at that word, after that layer. Think of it as the word's whole notebook.
2. **The attention output**: what that word copied from earlier words in that layer.
3. **The MLP output**: what that word looked up in memory in that layer (the value).

**The experiment** has three steps:

1. **Clean run.** Ask the question normally. The model answers correctly.
2. **Break it.** Add random noise to the country's name, so the model can no longer tell which country is meant. The correct answer's probability collapses.
3. **Repair one piece.** Run the broken sentence again, but copy back one single cell from the clean run. If the answer comes back, that piece was carrying the fact.

We measure each repair as **the percentage of the answer restored**: 0% means no help at all, 100% means the model is as confident as in the clean run.

### Engine: causal tracing

**What this cell does.** Defines the three steps of the experiment (clean run, break, repair) and runs the full experiment once, averaged over 60 countries, to choose a default layer for the MEMIT edit in Part 4.

**What you will see.** One line naming the default layer. The real exploration happens in the three labs below.

**Where the results are saved.** `memit_hw_data/causal_trace_default.csv` (every repair result from this default run).

In [ ]:
#@title Engine: causal tracing { display-mode: "form" }
KINDS = ["resid", "attn", "mlp"]
KIND_NAMES = {"resid": "Residual stream", "attn": "Attention output", "mlp": "MLP output"}
KIND_CMAPS = {"resid": "Purples", "attn": "Greens", "mlp": "Oranges"}

def _module(m, kind, l):
    return {"resid": m.blocks[l], "attn": m.blocks[l].attn, "mlp": m.blocks[l].mlp_out}[kind]

def trace_templates(relation):
    """Phrasings where the country is not the first word, so there are words before and after it."""
    return [t for t in TEMPLATES[relation]["train"] if not t.startswith("{s}")]

@torch.no_grad()
def trace_setup(m, subjects, relation, template, n_noise=10, noise_mult=3.0):
    """Steps 1 and 2: the clean run (recording every cell) and the broken run."""
    prompts = [fill(template, s) for s in subjects]
    answers = torch.tensor([stoi[facts[s][relation]] for s in subjects], device=DEVICE)
    ids, _ = encode(prompts)
    B, T = ids.shape
    L = len(m.blocks)
    subj_pos = template.split().index("{s}") + 1
    clean = {}
    def recorder(key):
        def fn(mod, inp, out):
            clean[key] = out.detach().clone()
        return fn
    with hooks([(_module(m, k, l), recorder((k, l))) for l in range(L) for k in KINDS]):
        clean_logits = m(ids)[:, -1]
    p_clean = clean_logits.softmax(-1)[torch.arange(B, device=DEVICE), answers].mean().item()

    ids_rep, ans_rep = ids.repeat(n_noise, 1), answers.repeat(n_noise)
    g = torch.Generator().manual_seed(SEED)
    noise = torch.zeros(B * n_noise, T, m.d_model)
    noise[:, subj_pos] = torch.randn(B * n_noise, m.d_model, generator=g) * noise_mult * m.tok.weight.std().item()
    noise = noise.to(DEVICE)
    broken_probs = m(ids_rep, embed_noise=noise)[:, -1].softmax(-1)
    p_broken = broken_probs[torch.arange(len(ans_rep), device=DEVICE), ans_rep].mean().item()
    labels = ["<bos>"] + [("[" + subjects[0] + "]" if B == 1 else "[country]") if w == "{s}" else w for w in template.split()]
    return {"m": m, "subjects": subjects, "relation": relation, "template": template, "ids_rep": ids_rep,
            "ans_rep": ans_rep, "noise": noise, "clean": clean, "B": B, "T": T, "L": L, "n_noise": n_noise,
            "subj_pos": subj_pos, "p_clean": p_clean, "p_broken": p_broken, "labels": labels,
            "clean_top": itos[clean_logits[0].argmax().item()],
            "broken_top": itos[broken_probs.mean(0).argmax().item()]}

@torch.no_grad()
def repair(ctx, kind, l, pos):
    """Step 3: rerun the broken sentence with ONE clean cell copied back. Returns P(correct answer)."""
    clean_val = ctx["clean"][(kind, l)][:, pos].repeat(ctx["n_noise"], 1)
    def patch(mod, inp, out):
        out = out.clone(); out[:, pos] = clean_val
        return out
    with hooks([(_module(ctx["m"], kind, l), patch)]):
        probs = ctx["m"](ctx["ids_rep"], embed_noise=ctx["noise"])[:, -1].softmax(-1)
    return probs[torch.arange(len(ctx["ans_rep"]), device=DEVICE), ctx["ans_rep"]].mean().item()

def restored(ctx, p):
    """Fraction of the answer restored: 0 = no help, 1 = as good as the clean run."""
    return (p - ctx["p_broken"]) / max(1e-6, ctx["p_clean"] - ctx["p_broken"])

def full_map(ctx):
    maps = {k: np.zeros((ctx["T"], ctx["L"])) for k in KINDS}
    for k in KINDS:
        for l in range(ctx["L"]):
            for pos in range(ctx["T"]):
                maps[k][pos, l] = restored(ctx, repair(ctx, k, l, pos))
    return maps

def maps_to_csv(ctx, maps, path):
    pd.DataFrame([{"component": KIND_NAMES[k], "layer": l + 1, "word": ctx["labels"][pos], "fraction restored": round(maps[k][pos, l], 4)}
                  for k in KINDS for l in range(ctx["L"]) for pos in range(ctx["T"])]).to_csv(path, index=False)

_default_ctx = trace_setup(base_model, known_countries[:60], "capital", trace_templates("capital")[0], n_noise=5)
_default_maps = full_map(_default_ctx)
maps_to_csv(_default_ctx, _default_maps, f"{DATA_DIR}/causal_trace_default.csv")
SUGGESTED_LAYER = int(np.argmax(_default_maps["mlp"][_default_ctx["subj_pos"]]))
print(f"Averaged over 60 countries, the MLP at layer {SUGGESTED_LAYER + 1} on the country's name matters most.")
print(f"Part 4 will edit layer {SUGGESTED_LAYER + 1} by default. (Layers are numbered 1 to {len(base_model.blocks)}.)")

TRACE_CTX, MANUAL_MAP, ON_TRACE_CHANGE = None, {}, []

def draw_grid(ctx, filled, selected=None):
    """The grid of cells: rows = words, columns = layers, three small squares per cell (R, A, M)."""
    T, L = ctx["T"], ctx["L"]
    fig, ax = plt.subplots(figsize=(1.5 * L + 2.5, 0.5 * T + 1.3))
    for pos in range(T):
        for l in range(L):
            for j, kind in enumerate(KINDS):
                val = filled.get((kind, l, pos))
                color = plt.get_cmap(KIND_CMAPS[kind])(0.15 + 0.85 * min(max(val, 0), 1)) if val is not None else "#f0f0f0"
                sel = selected == (kind, l, pos)
                ax.add_patch(Rectangle((l + 0.05 + j * 0.3, pos + 0.12), 0.28, 0.76, facecolor=color,
                                       edgecolor="black" if sel else "#b0b0b0", linewidth=2.5 if sel else 0.6))
                if val is not None:
                    ax.text(l + 0.19 + j * 0.3, pos + 0.5, f"{max(val, 0):.0%}", ha="center", va="center", fontsize=6.5,
                            color="white" if val > 0.6 else "black")
    ax.set_xlim(0, L); ax.set_ylim(T, 0)
    ax.set_xticks([l + 0.5 for l in range(L)]); ax.set_xticklabels([f"layer {l + 1}\nR  A  M" for l in range(L)], fontsize=8)
    ax.set_yticks([p + 0.5 for p in range(T)]); ax.set_yticklabels(ctx["labels"])
    ax.set_title("Your repair map   (R = residual stream, A = attention, M = MLP; grey = not tried yet)", fontsize=10)
    for s in ax.spines.values():
        s.set_visible(False)
    plt.tight_layout(); plt.show()

### Lab 3.1: ask the question, then break it

**What this cell does.** Runs steps 1 and 2 of the experiment for one country: the clean run, then the broken run with noise added to the country's name. The noise is repeated 10 times with different random values and the results are averaged.

**What you will see.** The sentence split into words, with the country highlighted. Then the model's confidence in the correct answer before and after breaking it.

**Try this.**

1. Run it with the default country. The confidence should fall from near 100% to near 0%.
2. Try a different phrasing or relation. Does breaking the country's name always destroy the answer? Why would it, when the rest of the sentence is untouched?

In [ ]:
#@title Lab 3.1: ask the question, then break it { display-mode: "form" }
t_country = widgets.Dropdown(options=sorted(known_countries), description="Country")
t_rel = widgets.Dropdown(options=RELATIONS, description="Relation")
t_tmpl = widgets.Dropdown(description="Phrasing", layout=widgets.Layout(width="420px"))
t_btn = widgets.Button(description="Run clean and broken", button_style="primary", layout=widgets.Layout(width="200px"))
t_out = widgets.Output()
t_rel.observe(lambda *_: setattr(t_tmpl, "options", trace_templates(t_rel.value)), "value")
t_tmpl.options = trace_templates(t_rel.value)

def word_chips(ctx):
    chips = ""
    for i, w in enumerate(ctx["labels"]):
        style = "background:#fff4c2;border:2px solid #c99a00;font-weight:bold" if i == ctx["subj_pos"] else "background:#eef1f5;border:1px solid #c0c6ce"
        chips += f'<span style="{style};border-radius:6px;padding:4px 8px;margin:2px;display:inline-block">{w}</span>'
    return chips + ' <span style="padding:4px 8px">...?</span>'

def run_clean_broken(*_):
    global TRACE_CTX, MANUAL_MAP
    with t_out:
        clear_output(wait=True)
        TRACE_CTX = trace_setup(base_model, [t_country.value], t_rel.value, t_tmpl.value, n_noise=10)
        MANUAL_MAP = {}
        c = TRACE_CTX
        correct = facts[t_country.value][t_rel.value]
        display(HTML(f'<div style="font-size:14px;margin:6px 0">{word_chips(c)}</div>'
                     f'<div style="font-size:14px"><b>1. Clean run.</b> The model answers <b>{c["clean_top"]}</b> '
                     f'and gives the correct answer <b>{correct}</b> a probability of <b>{c["p_clean"]:.0%}</b>.<br>'
                     f'<b>2. Broken run.</b> After adding noise to <b>{t_country.value}</b>, the correct answer drops to '
                     f'<b style="color:{RED}">{c["p_broken"]:.0%}</b>. The model now leans toward <b>{c["broken_top"]}</b>.</div>'))
        fig, ax = plt.subplots(figsize=(6, 1.8))
        ax.barh(["broken", "clean"], [c["p_broken"], c["p_clean"]], color=[RED, GREEN])
        ax.set_xlim(0, 1); ax.set_xlabel(f"P('{correct}')")
        plt.tight_layout(); plt.show()
        print("Now go to Lab 3.2 and try to repair it.")
        for fn in ON_TRACE_CHANGE:
            fn()

t_btn.on_click(run_clean_broken)
display(widgets.VBox([widgets.HBox([t_country, t_rel]), widgets.HBox([t_tmpl, t_btn]), t_out]))
run_clean_broken()

### Lab 3.2: repair one piece by hand

**What this cell does.** Runs step 3 for a single cell that you choose: the broken sentence is run again, but one piece (residual stream, attention output, or MLP output, at one layer and one word) is copied back from the clean run.

**What you will see.** How much of the answer that single repair brought back, a bar chart comparing clean, broken, and repaired, and your **repair map**, which colors in every cell you have tried so far. Darker means more of the answer came back.

**Try this.**

1. Choose **Residual stream**, the country's word, and layer 1. Repairing the country's whole notebook right at the start should fix nearly everything.
2. Keep **Residual stream** and move along the country's row, then along the last word's row. Where does the information sit early on, and where does it end up?
3. Switch to **MLP output** on the country's word and try each layer. Which layer's memory lookup restores the most?
4. Switch to **Attention output** on the last word and try each layer. This is where the fact is carried to the answer.

In [ ]:
#@title Lab 3.2: repair one piece by hand { display-mode: "form" }
r_kind = widgets.ToggleButtons(options=[("Residual stream", "resid"), ("Attention output", "attn"), ("MLP output", "mlp")], value="mlp")
r_layer = widgets.IntSlider(value=1, min=1, max=len(base_model.blocks), description="Layer")
r_word = widgets.Dropdown(description="Word")
r_btn = widgets.Button(description="Repair this piece", button_style="success")
r_clear = widgets.Button(description="Clear map")
r_out = widgets.Output()

def sync_words():
    r_word.options = [(w, i) for i, w in enumerate(TRACE_CTX["labels"])]
    r_word.value = TRACE_CTX["subj_pos"]
    with r_out:
        clear_output()
        print(f"Ready: {TRACE_CTX['labels'][TRACE_CTX['subj_pos']]} is broken. Pick a piece to repair.")
ON_TRACE_CHANGE.append(sync_words)
sync_words()

def do_repair(*_):
    with r_out:
        clear_output(wait=True)
        c, kind, l, pos = TRACE_CTX, r_kind.value, r_layer.value - 1, r_word.value
        p = repair(c, kind, l, pos)
        frac = restored(c, p)
        MANUAL_MAP[(kind, l, pos)] = frac
        display(HTML(f'<div style="font-size:14px">Repairing the <b>{KIND_NAMES[kind].lower()}</b> at <b>layer {l + 1}</b> '
                     f'on the word <b>{c["labels"][pos]}</b> brought back <b style="color:{GREEN if frac > 0.5 else NAVY}">{max(frac, 0):.0%}</b> '
                     f'of the answer (probability {p:.0%}).</div>'))
        fig, ax = plt.subplots(figsize=(6, 1.9))
        ax.barh(["repaired", "broken", "clean"], [p, c["p_broken"], c["p_clean"]], color=[GOLD, RED, GREEN])
        ax.set_xlim(0, 1); ax.set_xlabel("P(correct answer)")
        plt.tight_layout(); plt.show()
        draw_grid(c, MANUAL_MAP, selected=(kind, l, pos))

def clear_map(*_):
    MANUAL_MAP.clear()
    with r_out:
        clear_output(); draw_grid(TRACE_CTX, MANUAL_MAP)

r_btn.on_click(do_repair); r_clear.on_click(clear_map)
display(widgets.VBox([r_kind, widgets.HBox([r_layer, r_word]), widgets.HBox([r_btn, r_clear]), r_out]))

### Lab 3.3: fill in the whole map

**What this cell does.** Repairs every cell automatically, for all three kinds of pieces, and draws the complete maps you were building by hand. You can do this for the country from Lab 3.1 only, or averaged over many countries, which gives a smoother and more reliable picture.

**What you will see.** Three heatmaps, one per kind of piece, with the percentage of the answer restored in each cell. The strongest cells are boxed and labeled. Below the maps, a plain-language summary of what your model is doing.

**Where the results are saved.** `memit_hw_data/causal_trace_one_country.csv` or `memit_hw_data/causal_trace_average.csv`.

**Try this.**

1. Compare the map for one country with the average map. Are the bright regions in the same places?
2. Find the bright region in the MLP map and the bright region in the attention map. Together they tell the story of how the model recalls a fact.

In [ ]:
#@title Lab 3.3: fill in the whole map { display-mode: "form" }
f_mode = widgets.ToggleButtons(options=["This country only", "Average over many countries"], value="Average over many countries")
f_n = widgets.IntSlider(value=60, min=20, max=150, step=10, description="Countries")
f_btn = widgets.Button(description="Fill the whole map", button_style="primary")
f_out = widgets.Output()

def box(ax, pos, l, text, color):
    ax.add_patch(Rectangle((l - 0.5, pos - 0.5), 1, 1, fill=False, edgecolor=color, linewidth=2.5))
    ax.annotate(text, xy=(l, pos), xytext=(0.5, -0.28), textcoords="axes fraction", ha="center", fontsize=9, color=color,
                arrowprops=dict(arrowstyle="->", color=color))

def fill_map(*_):
    with f_out:
        clear_output(wait=True)
        print("Repairing every cell...")
        base = TRACE_CTX
        if f_mode.value == "This country only":
            ctx = base
            name, path = f"the country {base['subjects'][0]}", f"{DATA_DIR}/causal_trace_one_country.csv"
        else:
            ctx = trace_setup(base_model, known_countries[:f_n.value], base["relation"], base["template"], n_noise=5)
            name, path = f"{f_n.value} countries (average)", f"{DATA_DIR}/causal_trace_average.csv"
        maps = full_map(ctx)
        maps_to_csv(ctx, maps, path)
        clear_output(wait=True)
        s, last = ctx["subj_pos"], ctx["T"] - 1
        fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
        for ax, kind in zip(axes, KINDS):
            im = ax.imshow(np.clip(maps[kind], 0, 1), cmap=KIND_CMAPS[kind], vmin=0, vmax=1, aspect="auto")
            ax.set_title(f"Repair the {KIND_NAMES[kind].lower()}", fontsize=11)
            ax.set_yticks(range(ctx["T"])); ax.set_yticklabels(ctx["labels"])
            ax.set_xticks(range(ctx["L"])); ax.set_xticklabels(range(1, ctx["L"] + 1)); ax.set_xlabel("layer")
            cb = plt.colorbar(im, ax=ax, fraction=0.046); cb.set_label("answer restored")
            cb.set_ticks([0, 0.5, 1]); cb.set_ticklabels(["0%", "50%", "100%"])
        lm, la = int(np.argmax(maps["mlp"][s])), int(np.argmax(maps["attn"][last]))
        box(axes[2], s, lm, "memory lookup on the country's name", "#8a4b00")
        box(axes[1], last, la, "attention carries the fact to the last word", GREEN)
        fig.suptitle(f"Where the fact lives: {name}", fontsize=13)
        plt.tight_layout(); plt.show()

        resid_subj = maps["resid"][s]
        handoff = next((l for l in range(ctx["L"]) if resid_subj[l] < 0.5), None)
        handoff_text = f"and stops working from layer {handoff + 1}" if handoff is not None else "in every layer"
        mlp_pct, attn_pct = max(maps["mlp"][s, lm], 0), max(maps["attn"][last, la], 0)
        summary = (
            '<div style="font-size:14px;line-height:1.55;background:#fbfaf6;border-left:5px solid #c99a00;padding:10px 14px">'
            '<b>How to read these maps.</b> Each square answers one question: if we repair only this piece, how much of the answer comes back? '
            'Dark squares carry the fact. Light squares do not.<br><br><b>What your model is doing:</b><ol>'
            f"<li><b>Residual stream (left).</b> Repairing the country's word works in the early layers {handoff_text}. "
            "Repairing the last word works in the late layers. The information starts at the country's name and moves to the last word.</li>"
            f"<li><b>MLP output (right).</b> The strongest memory lookup is at <b>layer {lm + 1}</b> on the country's name, "
            f"restoring <b>{mlp_pct:.0%}</b> of the answer. This is where the fact is stored, and where MEMIT will edit it.</li>"
            f"<li><b>Attention output (middle).</b> The strongest attention is at <b>layer {la + 1}</b> on the last word, "
            f"restoring <b>{attn_pct:.0%}</b>. This is attention reading the fact from the country's position "
            "and bringing it to the word where the answer is predicted.</li></ol>"
            "This is the same two-stage story the MEMIT paper found in large models: MLPs look up facts about the subject, "
            "then attention moves them to where they are needed. In a small model the picture can be messier, and that is fine to notice.</div>")
        display(HTML(summary))

f_btn.on_click(fill_map)
display(widgets.VBox([f_mode, widgets.HBox([f_n, f_btn]), f_out]))

---
# Part 4. The MEMIT edit lab: see a fact in memory and rewrite it

This is the heart of the homework. You will pick one fact, find it inside the model, open the memory where it is stored, and rewrite it, one stage at a time.

| Stage | What happens |
|---|---|
| 1. Pick a fact | Choose a country, a relation, and the new answer you want the model to believe. |
| 2. Find where it lives | Run causal tracing on that one fact to choose the layer to edit. |
| 3. Open the memory | Look at the key, the memory matrix, and the value for your fact. |
| 4. Compute the new value | Find the value that would make the model give the new answer, without changing any weights yet. |
| 5. Write it into memory | Change the memory matrix with one formula, and see how small and precise the change is. |
| 6. Test the edit | Check the new fact, its paraphrases, and everything that should not have changed. |

The formula used in stage 5 is

$$\Delta W = R\,K^{\top}\left(\lambda C_0 + K K^{\top}\right)^{-1}$$

1. $K$ holds the keys for your fact (one column per phrasing).
2. $R$ holds the change we want in the layer's output (from stage 4).
3. $C_0$ describes what keys look like in ordinary text. It protects everything else the model knows.
4. $\lambda$ sets how strongly that protection counts.

Run each stage in order. Every stage builds on the previous one. If you change your fact in stage 1, run the later stages again.

### Engine: MEMIT

**What this cell does.** Defines every piece of MEMIT: reading keys, computing the covariance $C_0$, finding the new value, solving for the weight change, and the tests used to check an edit. It also defines the fine-tuning method we compare against.

**What you will see.** Nothing. The stages below use these functions.

**Where the results are saved.** Every edit you make is added to `memit_hw_data/edit_log.csv`.

In [ ]:
#@title Engine: MEMIT { display-mode: "form" }
def edit_prompts(request, which="train"):
    return [fill(t, request["subject"]) for t in TEMPLATES[request["relation"]][which]]

@torch.no_grad()
def get_keys(m, layer, prompts, subjects):
    """KEY: the input to mlp_out at the subject's word. One row per prompt."""
    store = {}
    def grab(mod, inp, out):
        store["k"] = inp[0].detach()
    ids, _ = encode(prompts)
    with hooks([(m.blocks[layer].mlp_out, grab)]):
        m(ids)
    return store["k"][torch.arange(len(prompts), device=DEVICE), subject_positions(prompts, subjects)]

@torch.no_grad()
def get_resid(m, layer, prompts, subjects):
    """The residual stream after a block, at the subject's word. One row per prompt."""
    store = {}
    def grab(mod, inp, out):
        store["h"] = out.detach()
    ids, _ = encode(prompts)
    with hooks([(m.blocks[layer], grab)]):
        m(ids)
    return store["h"][torch.arange(len(prompts), device=DEVICE), subject_positions(prompts, subjects)]

COV_CACHE = {}
@torch.no_grad()
def key_covariance(layer, batch_size=512):
    """C0 = the average of k k^T over every word of the training text, on the ORIGINAL model."""
    if layer in COV_CACHE:
        return COV_CACHE[layer]
    total, count = None, 0
    for i in range(0, len(train_items), batch_size):
        ids, _ = encode([p + " " + a for p, a in train_items[i:i + batch_size]])
        store = {}
        def grab(mod, inp, out):
            store["k"] = inp[0].detach()
        with hooks([(base_model.blocks[layer].mlp_out, grab)]):
            base_model(ids)
        K = store["k"][ids != PAD].double()
        total = K.T @ K if total is None else total + K.T @ K
        count += K.shape[0]
    COV_CACHE[layer] = total / count
    return COV_CACHE[layer]

def optimize_target(m, requests, layer, steps=100, lr=0.1, weight_decay=1e-3, max_norm_mult=4.0):
    """Find one vector per request that, added to the residual stream at `layer` (subject's word),
    makes the model give the new answer. All model weights stay frozen."""
    prompts, subjects, new_ids, old_ids, owner = [], [], [], [], []
    for j, r in enumerate(requests):
        ps = edit_prompts(r, "train")
        prompts += ps; subjects += [r["subject"]] * len(ps)
        new_ids += [stoi[r["new"]]] * len(ps); old_ids += [stoi[r["old"]]] * len(ps); owner += [j] * len(ps)
    ids, lengths = encode(prompts)
    pos = subject_positions(prompts, subjects)
    new_ids, old_ids = torch.tensor(new_ids, device=DEVICE), torch.tensor(old_ids, device=DEVICE)
    owner, rows = torch.tensor(owner, device=DEVICE), torch.arange(len(prompts), device=DEVICE)
    h0 = get_resid(m, layer, prompts, subjects)
    max_norm = max_norm_mult * h0.norm(dim=-1).mean()
    delta = torch.zeros(len(requests), m.d_model, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    def add_delta(mod, inp, out):
        out = out.clone(); out[rows, pos] = out[rows, pos] + delta[owner]
        return out
    log = []
    for step in range(steps):
        with hooks([(m.blocks[layer], add_delta)]):
            logits = m(ids)
        logp = logits[rows, lengths - 1].log_softmax(-1)
        nll = -logp[rows, new_ids].mean()
        loss = nll + weight_decay * (delta ** 2).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            n = delta.norm(dim=-1, keepdim=True)
            delta.mul_(torch.clamp(max_norm / (n + 1e-8), max=1.0))
        probs = logp.exp().detach()
        log.append({"step": step, "P(new answer)": probs[rows, new_ids].mean().item(),
                    "P(old answer)": probs[rows, old_ids].mean().item(),
                    "model says": itos[probs[0].argmax().item()]})
        if log[-1]["P(new answer)"] > 0.99:
            break
    delta = delta.detach()
    return {"prompts": prompts, "subjects": subjects, "h0": h0, "delta": delta, "layer": layer,
            "target": h0 + delta[owner], "log": pd.DataFrame(log)}

def solve_update(K, R, C0, lam):
    """Delta W = R K^T (lam C0 + K K^T)^-1.  K: (d_mlp, n) keys as columns, R: (d_model, n)."""
    K, R = K.double(), R.double()
    A = lam * C0 + K @ K.T + 1e-6 * torch.eye(K.shape[0], device=K.device, dtype=K.dtype)
    return torch.linalg.solve(A, K @ R.T).T.float()

def memit_edit(m, requests, layers, lam=20.0, steps=100, tgt=None, return_info=False):
    """MEMIT. With several layers, each layer takes an equal share of what is still missing."""
    edited = copy.deepcopy(m)
    last = layers[-1]
    if tgt is None or tgt["layer"] != last:
        tgt = optimize_target(edited, requests, last, steps=steps)
    info = {"target": tgt, "updates": [], "lam": lam, "layers": layers}
    for i, l in enumerate(layers):
        K = get_keys(edited, l, tgt["prompts"], tgt["subjects"]).T
        cur = get_resid(edited, last, tgt["prompts"], tgt["subjects"])
        R = ((tgt["target"] - cur) / (len(layers) - i)).T
        dW = solve_update(K, R, key_covariance(l), lam)
        W = edited.blocks[l].mlp_out.weight
        if i == 0:
            info["dW"], info["W"], info["K"], info["first_layer"] = dW.clone(), W.detach().clone(), K.T.clone(), l
        sv = torch.linalg.svdvals(dW)
        info["updates"].append({"layer": l + 1, "size of change (relative to the memory)": f"{(dW.norm() / W.norm()).item():.3%}",
                                "rank of the change": int((sv > 1e-3 * sv[0]).sum().item()),
                                "phrasings written": K.shape[1]})
        with torch.no_grad():
            W += dW
    after = get_resid(edited, last, tgt["prompts"], tgt["subjects"])
    wanted = tgt["target"] - tgt["h0"]
    info["achieved"] = (((after - tgt["h0"]) * wanted).sum(-1) / (wanted ** 2).sum(-1)).cpu().numpy()
    return (edited, info) if return_info else edited

def finetune_edit(m, requests, lr=1e-3, max_steps=300):
    """Comparison method: fine-tune ALL weights until every training phrasing gives the new answer."""
    ft = copy.deepcopy(m)
    for p in ft.parameters():
        p.requires_grad_(True)
    items = [(p, r["new"]) for r in requests for p in edit_prompts(r, "train")]
    ids, lengths = encode([p for p, _ in items])
    tg = torch.tensor([stoi[a] for _, a in items], device=DEVICE)
    rows = torch.arange(len(items), device=DEVICE)
    opt = torch.optim.Adam(ft.parameters(), lr=lr)
    ft.train()
    for step in range(max_steps):
        logits = ft(ids)[rows, lengths - 1]
        loss = F.cross_entropy(logits, tg)
        opt.zero_grad(); loss.backward(); opt.step()
        if (logits.argmax(-1) == tg).all():
            break
    ft.eval()
    for p in ft.parameters():
        p.requires_grad_(False)
    return ft

def make_requests(n, relation="capital", seed=0):
    rng = random.Random(SEED + 100 + seed)
    reqs = []
    for s in rng.sample(known_countries, n):
        old = facts[s][relation]
        reqs.append({"subject": s, "relation": relation, "old": old, "new": rng.choice([o for o in OBJECT_POOL[relation] if o != old])})
    return reqs

CHECK_NAMES = ["Efficacy", "Generalization", "Specificity: other countries", "Specificity: same country, other facts"]
def evaluate(m, requests, n_neighbors=100):
    """The three checks, in percent."""
    edited = {r["subject"] for r in requests}
    eff = [(p, r["new"]) for r in requests for p in edit_prompts(r, "train")]
    gen = [(p, r["new"]) for r in requests for p in edit_prompts(r, "heldout")]
    others = random.Random(SEED + 7).sample([c for c in known_countries if c not in edited], min(n_neighbors, len(known_countries) - len(edited)))
    rels = sorted({r["relation"] for r in requests})
    neigh = [(fill(t, c), facts[c][rel]) for c in others for rel in rels for t in TEMPLATES[rel]["train"]]
    same = [(fill(t, r["subject"]), facts[r["subject"]][r2]) for r in requests for r2 in RELATIONS if r2 != r["relation"] for t in TEMPLATES[r2]["train"]]
    return dict(zip(CHECK_NAMES, [100 * accuracy(m, x) for x in (eff, gen, neigh, same)]))

def log_edit(method, req, layers, lam, checks):
    row = {"time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "method": method, "country": req["subject"],
           "relation": req["relation"], "old answer": req["old"], "new answer": req["new"],
           "layers": " ".join(str(l + 1) for l in layers) if layers else "", "lambda": lam, **{k: round(v, 1) for k, v in checks.items()}}
    path = f"{DATA_DIR}/edit_log.csv"
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

def scorecard(before, after, req):
    """Before/after table colored green (as it should be) or red (not as it should be)."""
    s, rel = req["subject"], req["relation"]
    neighbor = next(c for c in known_countries if c != s)
    probes = [("The edited fact", edit_prompts(req)[0], req["new"]),
              ("Held-out phrasing 1", edit_prompts(req, "heldout")[0], req["new"]),
              ("Held-out phrasing 2", edit_prompts(req, "heldout")[1], req["new"]),
              ("Another country", fill(TEMPLATES[rel]["train"][0], neighbor), facts[neighbor][rel])]
    probes += [(f"Same country, {r2}", fill(TEMPLATES[r2]["train"][0], s), facts[s][r2]) for r2 in RELATIONS if r2 != rel]
    if rel == "capital":
        probes.append(("Reverse question", REVERSE_TEMPLATE.format(o=req["new"]), s))
    b, a = predict(before, [p for _, p, _ in probes]), predict(after, [p for _, p, _ in probes])
    html = ('<table style="border-collapse:collapse;font-size:13px"><tr style="background:#1f3a5f;color:white">'
            '<th style="padding:6px 10px">Check</th><th style="padding:6px 10px">Prompt</th><th style="padding:6px 10px">Ideal answer</th>'
            '<th style="padding:6px 10px">Before</th><th style="padding:6px 10px">After</th></tr>')
    for (name, p, ideal), x, y in zip(probes, b, a):
        ok = y == ideal
        html += (f'<tr><td style="padding:5px 10px;border-bottom:1px solid #ddd">{name}</td>'
                 f'<td style="padding:5px 10px;border-bottom:1px solid #ddd">{p} ...</td>'
                 f'<td style="padding:5px 10px;border-bottom:1px solid #ddd">{ideal}</td>'
                 f'<td style="padding:5px 10px;border-bottom:1px solid #ddd">{x}</td>'
                 f'<td style="padding:5px 10px;border-bottom:1px solid #ddd;background:{"#e6f0e9" if ok else "#f6e3e1"};'
                 f'color:{GREEN if ok else RED};font-weight:bold">{y} {"&#10003;" if ok else "&#10007;"}</td></tr>')
    return html + "</table>"

def check_bars(results, title):
    """results: {method name: evaluate(...) dict}"""
    df = pd.DataFrame(results).T[CHECK_NAMES]
    ax = df.T.plot.barh(figsize=(9, 2.2 + 0.5 * len(results)), color=[NAVY, GOLD, GREEN][:len(results)])
    ax.set_xlim(0, 105); ax.set_xlabel("percent"); ax.set_title(title); ax.invert_yaxis()
    ax.legend(fontsize=9, loc="lower right"); plt.tight_layout(); plt.show()

LAB = {}
def need(*keys):
    missing = [k for k in keys if k not in LAB]
    if missing:
        print("Please run the earlier stages first (missing: " + ", ".join(missing) + ").")
        return False
    return True

### Stage 1 of 6: pick a fact to rewrite

**What this cell does.** Lets you choose which fact to change. You pick a relation, a country, and the new answer you want the model to believe.

**What you will see.** A card with the current fact and the new fact, and a table of how confident the model is in the old and new answers on all 8 phrasings (6 training and 2 held out). Before editing, the old answer should be near 100% and the new one near 0%.

**Try this.**

1. Click **Random fact** for a quick start, or choose your own.
2. Try a capital first. Later, come back and try a language or currency edit.

In [ ]:
#@title Stage 1 of 6: pick a fact to rewrite { display-mode: "form" }
s1_rel = widgets.Dropdown(options=RELATIONS, description="Relation")
s1_country = widgets.Dropdown(options=sorted(known_countries), description="Country")
s1_new = widgets.Dropdown(description="New answer")
s1_random = widgets.Button(description="Random fact")
s1_go = widgets.Button(description="Choose this fact", button_style="primary")
s1_out = widgets.Output()

def s1_options(*_):
    old = facts[s1_country.value][s1_rel.value]
    s1_new.options = sorted(o for o in OBJECT_POOL[s1_rel.value] if o != old)
s1_rel.observe(s1_options, "value"); s1_country.observe(s1_options, "value"); s1_options()

def s1_pick_random(*_):
    s1_country.value = random.choice(known_countries)
    s1_new.value = random.choice(s1_new.options)
    s1_choose()

def s1_choose(*_):
    LAB.clear()
    req = {"subject": s1_country.value, "relation": s1_rel.value, "old": facts[s1_country.value][s1_rel.value], "new": s1_new.value}
    LAB["req"] = req
    with s1_out:
        clear_output(wait=True)
        example = edit_prompts(req)[0]
        display(HTML(f'<div style="font-size:15px;line-height:1.7;border:1px solid #d0d4da;border-radius:10px;padding:12px 16px;background:#fbfaf6">'
                     f'<b>Current fact:</b> {example} <span style="color:{RED};font-weight:bold">{req["old"]}</span><br>'
                     f'<b>New fact:</b> {example} <span style="color:{GREEN};font-weight:bold">{req["new"]}</span></div>'))
        prompts = edit_prompts(req) + edit_prompts(req, "heldout")
        display(pd.DataFrame({"phrasing": [p + " ..." for p in prompts],
                              "used in training?": ["yes"] * 6 + ["no (held out)"] * 2,
                              f"P('{req['old']}')": [f"{x:.0%}" for x in prob_of(base_model, prompts, req["old"])],
                              f"P('{req['new']}')": [f"{x:.0%}" for x in prob_of(base_model, prompts, req["new"])]}))
        print("Next: run Stage 2.")

s1_random.on_click(s1_pick_random); s1_go.on_click(s1_choose)
display(widgets.VBox([widgets.HBox([s1_rel, s1_country, s1_new]), widgets.HBox([s1_random, s1_go]), s1_out]))

### Stage 2 of 6: find where it lives

**What this cell does.** Runs causal tracing (Part 3) on your one fact, but only for the MLP outputs, to see which layer's memory holds it. It then chooses that layer for the edit. You can override the choice with the slider.

**What you will see.** A heatmap of how much of the answer each MLP restores, with the chosen layer boxed. Below it, the layer that will be edited.

**Try this.**

1. Compare the layer found for your fact with the default layer from Part 3 (averaged over many countries).
2. Choose **Spread over 3 layers** to use the full MEMIT method, which shares the change between neighboring layers.

In [ ]:
#@title Stage 2 of 6: find where it lives { display-mode: "form" }
s2_btn = widgets.Button(description="Find it", button_style="primary")
s2_layer = widgets.IntSlider(value=SUGGESTED_LAYER + 1, min=1, max=len(base_model.blocks), description="Edit layer")
s2_spread = widgets.ToggleButtons(options=[("One layer", 1), ("Spread over 3 layers", 3)], value=1)
s2_out, s2_choice = widgets.Output(), widgets.HTML()

def s2_update(*_):
    if "req" not in LAB:
        return
    L, n = s2_layer.value - 1, len(base_model.blocks)
    if s2_spread.value == 1:
        LAB["layers"] = [L]
    else:
        lo = min(max(0, L - 1), n - 3)
        LAB["layers"] = [lo, lo + 1, lo + 2]
    s2_choice.value = (f'<div style="font-size:14px">The edit will change the memory in <b>layer'
                       f'{"s" if len(LAB["layers"]) > 1 else ""} {", ".join(str(l + 1) for l in LAB["layers"])}</b>.</div>')
s2_layer.observe(s2_update, "value"); s2_spread.observe(s2_update, "value")

def s2_find(*_):
    if not need("req"):
        return
    req = LAB["req"]
    with s2_out:
        clear_output(wait=True)
        print("Tracing your fact...")
        ctx = trace_setup(base_model, [req["subject"]], req["relation"], trace_templates(req["relation"])[0], n_noise=10)
        row = np.array([restored(ctx, repair(ctx, "mlp", l, pos)) for pos in range(ctx["T"]) for l in range(ctx["L"])]).reshape(ctx["T"], ctx["L"])
        best = int(np.argmax(row[ctx["subj_pos"]]))
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7.5, 3.6))
        im = ax.imshow(np.clip(row, 0, 1), cmap="Oranges", vmin=0, vmax=1, aspect="auto")
        ax.set_yticks(range(ctx["T"])); ax.set_yticklabels(ctx["labels"])
        ax.set_xticks(range(ctx["L"])); ax.set_xticklabels(range(1, ctx["L"] + 1)); ax.set_xlabel("layer")
        ax.add_patch(Rectangle((best - 0.5, ctx["subj_pos"] - 0.5), 1, 1, fill=False, edgecolor="black", linewidth=3))
        ax.set_title(f"Which MLP holds the fact about {req['subject']}?")
        cb = plt.colorbar(im, ax=ax); cb.set_label("answer restored")
        plt.tight_layout(); plt.show()
        display(HTML(f'<div style="font-size:14px">For this fact, the MLP at <b>layer {best + 1}</b> on the word <b>{req["subject"]}</b> '
                     f'restores <b>{max(row[ctx["subj_pos"], best], 0):.0%}</b> of the answer. '
                     f'(Averaged over many countries in Part 3, the best layer was {SUGGESTED_LAYER + 1}.)</div>'))
    s2_layer.value = best + 1
    s2_update()
    with s2_out:
        print("Next: run Stage 3.")

s2_btn.on_click(s2_find)
display(widgets.VBox([s2_btn, s2_out, widgets.HBox([s2_layer, s2_spread]), s2_choice]))

### Stage 3 of 6: open the memory

**What this cell does.** Looks inside the memory at the layer you chose. It reads the **key** for your country (one for each of the 6 training phrasings), compares it with the keys of other countries, and shows how the memory turns the key into a **value**.

**What you will see.** Four panels: (1) the key drawn as a strip of 512 colored numbers, next to the keys of two other countries; (2) a similarity map showing that a country's key stays similar across phrasings but differs from other countries, like an ID card; (3) the multiplication memory x key = value with the real sizes; and (4) which words the value pushes the model toward.

**Try this.**

1. In the similarity map, look for the bright block in the top-left corner: the same country in different phrasings.
2. In panel 4, find where the current answer ranks among all words. In small models the value is not always easy to read directly, and that is part of why MEMIT computes the new value by optimization in Stage 4.

In [ ]:
#@title Stage 3 of 6: open the memory { display-mode: "form" }
s3_btn = widgets.Button(description="Open the memory", button_style="primary")
s3_out = widgets.Output()

def s3_open(*_):
    if not need("req", "layers"):
        return
    req, l = LAB["req"], LAB["layers"][0]
    tmpl0 = TEMPLATES[req["relation"]]["train"][0]
    others = [c for c in known_countries if c != req["subject"]][:6]
    with s3_out:
        clear_output(wait=True)
        own = edit_prompts(req)
        K_own = get_keys(base_model, l, own, [req["subject"]] * len(own))
        K_oth = get_keys(base_model, l, [fill(tmpl0, c) for c in others], others)

        # Panel 1: the key as a strip of numbers
        fig, axes = plt.subplots(3, 1, figsize=(12, 2.4), sharex=True)
        strips = [(req["subject"], K_own[0]), (others[0], K_oth[0]), (others[1], K_oth[1])]
        vmax = max(float(k.abs().max()) for _, k in strips)
        for ax, (name, k) in zip(axes, strips):
            ax.imshow(k.cpu().numpy()[None, :], aspect="auto", cmap="magma", vmin=0, vmax=vmax)
            ax.set_yticks([0]); ax.set_yticklabels([name], fontsize=10); ax.tick_params(axis="x", labelsize=8)
        axes[0].set_title(f"(1) The key at layer {l + 1}: {K_own.shape[1]} numbers per country, for the phrasing \"{own[0]}\"", fontsize=11)
        axes[-1].set_xlabel("position in the key vector")
        plt.tight_layout(); plt.show()

        # Panel 2: keys work like ID cards
        allK = F.normalize(torch.cat([K_own, K_oth]), dim=-1)
        names = [p.replace(req["subject"], req["subject"].upper()) for p in own] + [fill(tmpl0, c) for c in others]
        fig, ax = plt.subplots(figsize=(8.5, 6.2))
        im = ax.imshow((allK @ allK.T).cpu().numpy(), cmap="Blues", vmin=0, vmax=1)
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
        ax.set_xticks(range(len(names))); ax.set_xticklabels(range(1, len(names) + 1), fontsize=8)
        ax.axhline(len(own) - 0.5, color=GOLD, linewidth=2); ax.axvline(len(own) - 0.5, color=GOLD, linewidth=2)
        ax.set_title(f"(2) How similar are the keys?  Top-left block: {req['subject']} in 6 phrasings.  Bottom-right: other countries.", fontsize=10)
        plt.colorbar(im, ax=ax, fraction=0.046, label="cosine similarity")
        plt.tight_layout(); plt.show()

        # Panel 3: memory x key = value
        W = base_model.blocks[l].mlp_out.weight
        k = K_own[0]
        v = W @ k + base_model.blocks[l].mlp_out.bias
        show = lambda x: "[" + ", ".join(f"{float(t):.2f}" for t in x[:5]) + ", ...]"
        subj = req["subject"]
        box = "font-size:14px;line-height:1.8;border:1px solid #d0d4da;border-radius:10px;padding:12px 16px;background:#fbfaf6"
        html = (f"<div style='{box}'><b>(3) The lookup, with real numbers</b><br>"
                f"<span style='background:#fff4c2;padding:2px 6px;border-radius:4px'><b>memory</b> (layer {l + 1} mlp_out.weight): "
                f"{W.shape[0]} x {W.shape[1]} = {W.numel():,} numbers</span><br>"
                f"&times; <span style='background:#ece6f6;padding:2px 6px;border-radius:4px'><b>key</b> for {subj}: {show(k)} ({k.numel()} numbers)</span><br>"
                f"= <span style='background:#e6f0e9;padding:2px 6px;border-radius:4px'><b>value</b>: {show(v)} ({v.numel()} numbers)</span><br>"
                f"This one matrix stores facts about all {len(countries)} countries. MEMIT will change it so that <i>this</i> key gives a new value, "
                "while other keys keep giving their old values.</div>")
        display(HTML(html))

        # Panel 4: what does the value push toward?
        with torch.no_grad():
            logits = base_model.head(base_model.ln_f(v))
        order = logits.argsort(descending=True).tolist()
        rank = order.index(stoi[req["old"]]) + 1
        top = [itos[i] for i in order[:8]]
        display(HTML(f'<div style="font-size:14px;margin-top:8px"><b>(4) Reading the value.</b> If we send this value straight to the output layer, '
                     f'the words it pushes toward most are: <i>{", ".join(top)}</i>.<br>The current answer <b>{req["old"]}</b> ranks '
                     f'<b>#{rank}</b> out of {len(itos)} words.</div>'))
        print("\nNext: run Stage 4.")

s3_btn.on_click(s3_open)
display(widgets.VBox([s3_btn, s3_out]))

### Stage 4 of 6: compute the new value

**What this cell does.** Finds the value that would make the model give your new answer. It adds one adjustable vector to the residual stream at the subject's word and nudges that vector, step by step, until all 6 training phrasings give the new answer. **No weight of the model changes in this stage.** We only learn what the memory should output.

**What you will see.** A plot of the old answer's probability falling while the new answer's probability rises, and a table of what the model says at each step.

**Try this.**

1. Watch the point where the two lines cross. That is when the model switches its answer.
2. Compare the size of the vector with the size of the hidden state. How big a push is needed?

In [ ]:
#@title Stage 4 of 6: compute the new value { display-mode: "form" }
s4_btn = widgets.Button(description="Find the new value", button_style="primary")
s4_out = widgets.Output()

def s4_run(*_):
    if not need("req", "layers"):
        return
    req = LAB["req"]
    with s4_out:
        clear_output(wait=True)
        tgt = optimize_target(base_model, [req], LAB["layers"][-1])
        LAB["tgt"] = tgt
        log = tgt["log"]
        fig, ax = plt.subplots(figsize=(8, 3.4))
        ax.plot(log["step"], log["P(old answer)"], color=RED, linewidth=2.5, label=f"P('{req['old']}')  old answer")
        ax.plot(log["step"], log["P(new answer)"], color=GREEN, linewidth=2.5, label=f"P('{req['new']}')  new answer")
        ax.set_ylim(0, 1.02); ax.set_xlabel("optimization step"); ax.set_ylabel("probability (average of 6 phrasings)")
        ax.set_title("Finding the new value (all weights frozen)"); ax.legend()
        plt.tight_layout(); plt.show()
        idx = sorted(set(np.linspace(0, len(log) - 1, min(8, len(log))).astype(int)))
        t = log.iloc[idx].copy()
        t["P(old answer)"] = t["P(old answer)"].map("{:.0%}".format); t["P(new answer)"] = t["P(new answer)"].map("{:.0%}".format)
        display(t.rename(columns={"model says": f'model says ("{edit_prompts(req)[0]} ...")'}).reset_index(drop=True))
        print(f"\nFinished after {len(log)} steps. Size of the push: {tgt['delta'].norm().item():.2f}, "
              f"compared with a hidden state of size {tgt['h0'].norm(dim=-1).mean().item():.2f}.")
        print("We now know WHAT the memory should output. Stage 5 writes it into the weights.")

s4_btn.on_click(s4_run)
display(widgets.VBox([s4_btn, s4_out]))

### Stage 5 of 6: write it into memory

**What this cell does.** Uses the MEMIT formula to change the memory matrix so that your country's keys produce the new value, while $\lambda C_0$ protects all other keys. This is one calculation, not training.

**What you will see.** A table with the size and rank of the change; the memory matrix before, the change drawn on the same color scale (almost invisible), and the change magnified; and the most important plot: how much the change affects your country's keys compared with 50 other countries' keys.

**Try this.**

1. Look at the last plot. A precise edit gives tall bars for your country and bars near zero for everyone else.
2. Set λ to 0.1 and write the edit again. What happens to the bars for other countries? Then try 10000. Remember to finish with a value you like before Stage 6.

In [ ]:
#@title Stage 5 of 6: write it into memory { display-mode: "form" }
s5_lam = widgets.SelectionSlider(options=[0.1, 1, 10, 20, 100, 1000, 10000], value=20, description="λ")
s5_btn = widgets.Button(description="Write it in", button_style="success")
s5_out = widgets.Output()

def s5_write(*_):
    if not need("req", "layers", "tgt"):
        return
    req = LAB["req"]
    with s5_out:
        clear_output(wait=True)
        edited, info = memit_edit(base_model, [req], LAB["layers"], lam=s5_lam.value, tgt=LAB["tgt"], return_info=True)
        LAB.update({"edited": edited, "info": info, "lam": s5_lam.value})
        LAB.pop("finetuned", None)
        display(pd.DataFrame(info["updates"]).set_index("layer"))

        W, dW, l = info["W"].cpu().numpy(), info["dW"].cpu().numpy(), info["first_layer"]
        lim, dlim = np.abs(W).max(), np.abs(dW).max()
        fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
        for ax, M, vm, title in [(axes[0], W, lim, f"Memory before (layer {l + 1})"),
                                 (axes[1], dW, lim, "The change, same color scale"),
                                 (axes[2], dW, dlim, f"The change, magnified {lim / dlim:.0f}x")]:
            im = ax.imshow(M, cmap="RdBu_r", vmin=-vm, vmax=vm, aspect="auto")
            ax.set_title(title, fontsize=11); ax.set_xlabel("key position"); ax.set_ylabel("value position")
        plt.tight_layout(); plt.show()

        # How much does the change affect each key?
        others = [c for c in known_countries if c != req["subject"]][:50]
        tmpl0 = TEMPLATES[req["relation"]]["train"][0]
        K_oth = get_keys(base_model, l, [fill(tmpl0, c) for c in others], others)
        dWt = info["dW"]
        own_effect = (info["K"] @ dWt.T).norm(dim=-1).cpu().numpy()
        oth_effect = (K_oth @ dWt.T).norm(dim=-1).cpu().numpy()
        fig, ax = plt.subplots(figsize=(13, 3.4))
        ax.bar(range(len(own_effect)), own_effect, color=GREEN, label=f"{req['subject']} (6 phrasings)")
        ax.bar(range(len(own_effect), len(own_effect) + len(oth_effect)), oth_effect, color=GRAY, label="50 other countries")
        ax.set_ylabel("how much the output changes"); ax.set_xticks([])
        ax.set_title("Who does the change affect?  Tall bars = the value changed, bars near zero = untouched")
        ax.legend(); plt.tight_layout(); plt.show()
        print(f"Average effect on {req['subject']}: {own_effect.mean():.3f}.  Average effect on other countries: {oth_effect.mean():.3f} "
              f"({oth_effect.mean() / max(own_effect.mean(), 1e-9):.1%} of it).")
        print("\nNext: run Stage 6 to test the edited model.")

s5_btn.on_click(s5_write)
display(widgets.VBox([widgets.HBox([s5_lam, s5_btn]), s5_out]))

### Stage 6 of 6: test the edit

**What this cell does.** Tests the edited model on the three checks from the lecture: **efficacy** (the edited fact changed), **generalization** (the held-out phrasings changed too), and **specificity** (other countries and your country's other facts did not change). You can also run the same edit with ordinary fine-tuning and compare.

**What you will see.** A before and after table colored green where the model behaves as it should and red where it does not, a bar chart of the three checks, and a quiz where you can ask the original and edited models anything.

**Where the results are saved.** Each test is added to `memit_hw_data/edit_log.csv`, so you have a record of every edit and its results.

**Try this.**

1. Look at the reverse question row (for capital edits). Is it green or red? Think about which key MEMIT changed.
2. Click **Compare with fine-tuning**. Which method keeps other facts intact?
3. Go back to Stage 1, choose a different fact, and run the stages again.

In [ ]:
#@title Stage 6 of 6: test the edit { display-mode: "form" }
s6_btn = widgets.Button(description="Test the edit", button_style="primary")
s6_ft = widgets.Button(description="Compare with fine-tuning")
s6_out = widgets.Output()

def s6_test(*_):
    if not need("req", "edited"):
        return
    req = LAB["req"]
    with s6_out:
        clear_output(wait=True)
        display(HTML(scorecard(base_model, LAB["edited"], req)))
        checks = evaluate(LAB["edited"], [req])
        LAB["checks"] = checks
        log_edit("MEMIT", req, LAB["layers"], LAB["lam"], checks)
        check_bars({"MEMIT": checks}, "The three checks for your edit")
        print("Ask the original and edited models anything:")
        quiz_ui(lambda: {"Original": base_model, "After MEMIT": LAB["edited"],
                         **({"After fine-tuning": LAB["finetuned"]} if "finetuned" in LAB else {})})

def s6_compare(*_):
    if not need("req", "edited", "checks"):
        return
    req = LAB["req"]
    with s6_out:
        clear_output(wait=True)
        print("Fine-tuning all weights until the new fact is learned...")
        LAB["finetuned"] = finetune_edit(base_model, [req])
        ft_checks = evaluate(LAB["finetuned"], [req])
        log_edit("fine-tuning", req, [], "", ft_checks)
        clear_output(wait=True)
        display(HTML("<b>After MEMIT</b>" + scorecard(base_model, LAB["edited"], req)))
        display(HTML("<br><b>After fine-tuning</b>" + scorecard(base_model, LAB["finetuned"], req)))
        check_bars({"MEMIT": LAB["checks"], "Fine-tuning": ft_checks}, "MEMIT vs fine-tuning on the same edit")
        quiz_ui(lambda: {"Original": base_model, "After MEMIT": LAB["edited"], "After fine-tuning": LAB["finetuned"]})

s6_btn.on_click(s6_test); s6_ft.on_click(s6_compare)
display(widgets.VBox([widgets.HBox([s6_btn, s6_ft]), s6_out]))

---
# Part 5. Mass editing: from one fact to one hundred

MEMIT stands for **Mass-Editing** Memory in a Transformer. Its main advantage is editing many facts in one update. Here we change many capitals at the same time and compare three methods:

1. MEMIT on one layer
2. MEMIT spread over three layers (the full method from the paper)
3. Fine-tuning all weights until every new fact is learned

### Lab: mass editing

**What this cell does.** Edits 1, 10, 50, and 100 capitals at once with each of the three methods, then runs the three checks on every result.

**What you will see.** Four plots, one per check, showing how each method holds up as the number of edits grows, and a table with every number. This takes a few minutes.

**Where the results are saved.** `memit_hw_data/mass_editing_results.csv`.

**Try this.**

1. Look at efficacy for one-layer MEMIT at 100 edits. One layer's key space has 512 dimensions, and 100 edits with 6 phrasings each ask it to store 600 new key-value pairs.
2. Look at specificity for fine-tuning as the number of edits grows.

In [ ]:
#@title Lab: mass editing { display-mode: "form" }
m_sizes = widgets.SelectMultiple(options=[1, 5, 10, 25, 50, 100], value=[1, 10, 50, 100], description="Edits", rows=6)
m_btn = widgets.Button(description="Run", button_style="primary")
m_out = widgets.Output()

def three_layers(L):
    lo = min(max(0, L - 1), len(base_model.blocks) - 3)
    return [lo, lo + 1, lo + 2]

def run_mass(*_):
    with m_out:
        clear_output(wait=True)
        L, rows = SUGGESTED_LAYER, []
        for n in sorted(m_sizes.value):
            print(f"Editing {n} facts at once...")
            reqs = make_requests(n, seed=3)
            for name, m in [("MEMIT, one layer", memit_edit(base_model, reqs, [L])),
                            ("MEMIT, three layers", memit_edit(base_model, reqs, three_layers(L))),
                            ("Fine-tuning", finetune_edit(base_model, reqs))]:
                rows.append({"facts edited": n, "method": name, **evaluate(m, reqs)})
        df = pd.DataFrame(rows)
        df.round(1).to_csv(f"{DATA_DIR}/mass_editing_results.csv", index=False)
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), sharey=True)
        for ax, metric in zip(axes, CHECK_NAMES):
            for (name, g), color in zip(df.groupby("method", sort=False), [NAVY, GOLD, RED]):
                ax.plot(g["facts edited"], g[metric], marker="o", color=color, label=name)
            ax.set_xscale("log"); ax.set_title(metric, fontsize=10); ax.set_xlabel("facts edited at once"); ax.set_ylim(0, 102)
        axes[0].set_ylabel("percent"); axes[0].legend(fontsize=8)
        plt.tight_layout(); plt.show()
        display(df.round(1))

m_btn.on_click(run_mass)
display(widgets.VBox([widgets.HBox([m_sizes, m_btn]), m_out]))

---
# Part 6. Experiments: layers, protection strength, and limits

Four short experiments, each run from the same lab:

1. **Which layer to edit:** the same single-fact edits made at every layer.
2. **Protection strength λ:** the same edits with λ from very weak to very strong.
3. **One at a time vs all together:** edits applied one after another, compared with a single batch.
4. **Do edits ripple?** After changing a capital, ask the reverse question: "<new city> is the capital of".

### Lab: experiments

**What this cell does.** Runs the experiment you choose on the number of facts you choose.

**What you will see.** A plot or a table for that experiment. Each run takes from a few seconds to a few minutes.

**Where the results are saved.** `memit_hw_data/experiment_<name>.csv` for each experiment you run.

**Try this.**

1. Run **Which layer to edit** and compare the best layer with your causal tracing map from Part 3.
2. Run **Do edits ripple?** and explain the result to yourself using keys and values: which key did MEMIT change, and which key does the reverse question use?

In [ ]:
#@title Lab: experiments { display-mode: "form" }
x_exp = widgets.Dropdown(options=["Which layer to edit", "Protection strength λ", "One at a time vs all together", "Do edits ripple?"],
                         description="Experiment", layout=widgets.Layout(width="420px"))
x_n = widgets.IntSlider(value=15, min=5, max=50, step=5, description="Facts")
x_btn = widgets.Button(description="Run", button_style="primary")
x_out = widgets.Output()

def sweep_plot(df, title, xlabel, logx=False, mark=None):
    ax = df.plot(marker="o", logx=logx, figsize=(8.5, 4), color=[NAVY, GOLD, GREEN, RED])
    if mark is not None:
        ax.axvline(mark, color=GRAY, linestyle=":", label="layer found in Part 3")
    ax.set_ylim(0, 102); ax.set_ylabel("percent"); ax.set_xlabel(xlabel); ax.set_title(title)
    ax.legend(fontsize=8, loc="lower left"); plt.tight_layout(); plt.show()
    display(df.round(1))

def run_experiment(*_):
    with x_out:
        clear_output(wait=True)
        print("Running...")
        n, L, name = x_n.value, SUGGESTED_LAYER, x_exp.value
        if name == "Which layer to edit":
            reqs = make_requests(n, seed=1)
            df = pd.DataFrame([{"layer": l + 1, **pd.DataFrame([evaluate(memit_edit(base_model, [r], [l]), [r]) for r in reqs]).mean()}
                               for l in range(len(base_model.blocks))]).set_index("layer")
            clear_output(wait=True); sweep_plot(df, "Single-fact edits at each layer", "layer", mark=L + 1)
            df.round(1).to_csv(f"{DATA_DIR}/experiment_layers.csv")
        elif name == "Protection strength λ":
            reqs = make_requests(n, seed=2)
            df = pd.DataFrame([{"λ": lam, **pd.DataFrame([evaluate(memit_edit(base_model, [r], [L], lam=lam), [r]) for r in reqs]).mean()}
                               for lam in [0.1, 1, 10, 100, 1000, 10000]]).set_index("λ")
            clear_output(wait=True); sweep_plot(df, f"Effect of λ (layer {L + 1})", "λ", logx=True)
            df.round(1).to_csv(f"{DATA_DIR}/experiment_lambda.csv")
        elif name == "One at a time vs all together":
            reqs = make_requests(n, seed=4)
            batch = memit_edit(base_model, reqs, three_layers(L))
            seq = base_model
            for r in reqs:
                seq = memit_edit(seq, [r], three_layers(L))
            clear_output(wait=True)
            results = {"All together (one batch)": evaluate(batch, reqs), "One at a time": evaluate(seq, reqs)}
            check_bars(results, f"{n} edits: one batch vs one at a time")
            pd.DataFrame(results).T.round(1).to_csv(f"{DATA_DIR}/experiment_sequential.csv")
            display(pd.DataFrame(results).T.round(1))
        else:
            reqs = make_requests(n, seed=5)
            edited = memit_edit(base_model, reqs, three_layers(L))
            fwd = predict(edited, [edit_prompts(r)[0] for r in reqs])
            rev = [REVERSE_TEMPLATE.format(o=r["new"]) for r in reqs]
            before, after = predict(base_model, rev), predict(edited, rev)
            df = pd.DataFrame([{"edited country": r["subject"], "new capital": r["new"], "forward question after edit": f,
                                "reverse question": p, "answer before": b, "answer after": a,
                                "reverse gives edited country?": a == r["subject"]}
                               for r, f, p, b, a in zip(reqs, fwd, rev, before, after)])
            df.to_csv(f"{DATA_DIR}/experiment_ripple.csv", index=False)
            clear_output(wait=True)
            print(f"Forward edits that worked: {np.mean([f == r['new'] for f, r in zip(fwd, reqs)]):.0%}")
            print(f"Reverse questions that now give the edited country: {df['reverse gives edited country?'].mean():.0%}")
            display(df.head(10))

x_btn.on_click(run_experiment)
display(widgets.VBox([widgets.HBox([x_exp, x_n, x_btn]), x_out]))

---
# Part 7 (Bonus). MEMIT on GPT-2 XL

This optional part needs a GPU. It applies the same four MEMIT steps to a real pretrained model with 1.5 billion parameters. GPT-2 splits words into several tokens, so the key is read at the **last token of the subject**. If you do not have a GPU, skip to Part 8.

### Engine: load GPT-2 XL

**What this cell does.** Downloads GPT-2 XL (about 6 GB) and 2,000 paragraphs of Wikipedia text, which are used to compute $C_0$ for the real model. It also defines the same MEMIT steps as Part 4, adapted to GPT-2.

**What you will see.** A confirmation line when everything is loaded. This takes several minutes the first time.

**Where the results are saved.** The model is cached by the `transformers` library, not in `memit_hw_data/`.

In [ ]:
#@title Engine: load GPT-2 XL { display-mode: "form" }
!pip -q install transformers datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

assert DEVICE == "cuda", "Part 7 needs a GPU runtime."
tok = AutoTokenizer.from_pretrained("gpt2-xl")
tok.pad_token = tok.eos_token
gpt = AutoModelForCausalLM.from_pretrained("gpt2-xl").to(DEVICE).eval()
for p in gpt.parameters():
    p.requires_grad_(False)
wiki = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train[:30000]")
wiki_texts = [t for t in wiki["text"] if len(t.split()) > 40][:2000]
print(f"Loaded GPT-2 XL ({len(gpt.transformer.h)} layers) and {len(wiki_texts)} Wikipedia paragraphs.")

GPT_COV, GPT_ORIGINAL_W = {}, {}
def block_hidden(out):
    return out[0] if isinstance(out, tuple) else out
def with_hidden(out, h):
    return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h

@torch.no_grad()
def gpt_cov(layer):
    if layer in GPT_COV:
        return GPT_COV[layer]
    if len(GPT_COV) >= 2:
        GPT_COV.pop(next(iter(GPT_COV)))
    proj = gpt.transformer.h[layer].mlp.c_proj
    C, n, store = torch.zeros(proj.weight.shape[0], proj.weight.shape[0], device=DEVICE), 0, {}
    h = proj.register_forward_hook(lambda mod, inp, out: store.__setitem__("k", inp[0]))
    for i in range(0, len(wiki_texts), 8):
        enc = tok(wiki_texts[i:i + 8], return_tensors="pt", truncation=True, max_length=128, padding=True).to(DEVICE)
        gpt(**enc)
        k = store["k"][enc["attention_mask"].bool()].float()
        C += k.T @ k; n += k.shape[0]
    h.remove()
    GPT_COV[layer] = (C / n).double()
    return GPT_COV[layer]

@torch.no_grad()
def gpt_restore():
    for l, W in GPT_ORIGINAL_W.items():
        gpt.transformer.h[l].mlp.c_proj.weight.copy_(W)

@torch.no_grad()
def gpt_generate(prompt, n=6):
    enc = tok(prompt, return_tensors="pt").to(DEVICE)
    out = gpt.generate(**enc, max_new_tokens=n, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:])

def gpt_edit(subject, prompts, target, layer, lam, steps=30):
    gpt_restore()
    proj, block = gpt.transformer.h[layer].mlp.c_proj, gpt.transformer.h[layer]
    GPT_ORIGINAL_W.setdefault(layer, proj.weight.detach().clone())
    positions = [len(tok(p[:p.index(subject) + len(subject)])["input_ids"]) - 1 for p in prompts]
    n_t = len(tok(target)["input_ids"])
    keys = []
    for p, pos in zip(prompts, positions):                       # 1. keys
        store = {}
        h = proj.register_forward_hook(lambda mod, inp, out: store.__setitem__("k", inp[0].detach()))
        with torch.no_grad():
            gpt(**tok(p, return_tensors="pt").to(DEVICE))
        h.remove()
        keys.append(store["k"][0, pos])
    K1 = torch.stack(keys, dim=1)
    C0 = gpt_cov(layer)                                          # 2. covariance
    delta = torch.zeros(gpt.config.n_embd, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=0.5)
    log = []
    for step in range(steps):                                    # 3. new value
        loss = 0.0
        for p, pos in zip(prompts, positions):
            full = tok(p + target, return_tensors="pt").to(DEVICE)["input_ids"]
            def add(mod, inp, out, pos=pos):
                hs = block_hidden(out).clone(); hs[:, pos] = hs[:, pos] + delta
                return with_hidden(out, hs)
            hk = block.register_forward_hook(add)
            lp = gpt(full).logits[0, -n_t - 1:-1].log_softmax(-1)
            hk.remove()
            loss = loss + (-lp[torch.arange(n_t, device=DEVICE), full[0, -n_t:]].mean())
        loss = loss / len(prompts) + 1e-3 * delta.norm() ** 2
        opt.zero_grad(); loss.backward(); opt.step()
        log.append(loss.item())
    R = delta.detach().unsqueeze(1).repeat(1, K1.shape[1])        # 4. write it in
    dW = solve_update(K1, R, C0, lam)
    with torch.no_grad():
        proj.weight += dW.T                                      # GPT-2 stores this matrix transposed
    return log, (dW.norm() / GPT_ORIGINAL_W[layer].norm()).item()

### Lab: edit GPT-2 XL

**What this cell does.** It runs the same four MEMIT steps from Part 4 on the real GPT-2 XL model:

1. It reads the key for your **subject** at the chosen layer, from each edit prompt.
2. It uses the Wikipedia text to compute $C_0$ for that layer.
3. It finds the value that makes GPT-2 XL continue each edit prompt with your **new answer**.
4. It writes the change into that layer's memory matrix, then shows how GPT-2 XL continues each test prompt before and after the edit.

**What you will see.** A table of GPT-2 XL's continuations for your test prompts, before and after the edit. Each click of **Apply edit** starts from the original GPT-2 XL, so edits do not pile up. **Restore original** undoes the current edit.

**What each field means.**

1. **Subject:** the thing the fact is about. The key is read at the last token of this word, so it must appear exactly (same spelling and capitalization) in every edit prompt. The lab checks this and stops if it is missing.
2. **New answer:** what you want the model to say. The lab adds the leading space for you, and answers with several words, such as "New York", are fine.
3. **Edit prompts:** the sentences used to *write* the fact, one per line. Each one must stop right where the answer would come next. For example, with the defaults the lab teaches GPT-2 XL that "The capital of France is" continues with your new answer. These play the same role as the six training phrasings in Part 4.
4. **Test prompts:** the sentences used to *check* the edit, one per line. They are never used for writing, only for the before and after table.
5. **Layer and λ:** the same controls as in Part 4. Layer 17 is the layer the ROME paper edited in GPT-2 XL.

**When do you need to change the prompts?**

1. **Same subject, same kind of fact** (France's capital, any new city): change only the new answer. Everything else can stay.
2. **Same subject, a different kind of fact** (for example, France's official language): rewrite the edit prompts so they lead to that answer, such as "The official language of France is".
3. **A different subject** (for example, Lionel Messi): rewrite both the edit prompts and the test prompts.

**How to choose good test prompts.** Use the three checks from the homework:

1. **Prompts that should change** (efficacy and generalization): one of the edit prompts, plus a paraphrase that was not used for editing.
2. **Prompts that should not change** (specificity): the same question about a different subject, such as "The capital of Germany is".
3. **Other facts about the same subject** that should not change, such as "The official language of France is".

The default test prompts follow this pattern. Look closely at "The French government is based in the city of": it uses the word "French", not "France". MEMIT edited the key for the "France" token, so this prompt often does **not** change. That is the no-ripple limit from Part 6, now on a real model.

**Try this.**

1. Run the default edit and check which test prompts changed.
2. Change only the new answer (for example, to New York) and apply again.
3. Try a new subject. Replace the fields with this example, one prompt per line:
   1. **Subject:** Lionel Messi
   2. **New answer:** basketball
   3. **Edit prompts:** Lionel Messi plays the sport of / Lionel Messi is a professional player of / The sport Lionel Messi is famous for is
   4. **Test prompts:** Lionel Messi's sport is (should change) / Cristiano Ronaldo plays the sport of (should not change) / Lionel Messi was born in (should not change)
4. Change λ and compare how strong the edit is and how many other answers change.

In [ ]:
#@title Lab: edit GPT-2 XL { display-mode: "form" }
g_subject = widgets.Text(value="France", description="Subject")
g_target = widgets.Text(value="Lyon", description="New answer")
g_prompts = widgets.Textarea(value="The capital of France is\nFrance's capital city is\nThe capital city of France is",
                             description="Edit prompts", layout=widgets.Layout(width="540px", height="80px"))
g_tests = widgets.Textarea(value="The capital of France is\nThe French government is based in the city of\nThe capital of Germany is\nThe capital of Italy is\nThe official language of France is",
                           description="Test prompts", layout=widgets.Layout(width="540px", height="110px"))
g_layer = widgets.IntSlider(value=17, min=5, max=40, description="Layer")
g_lam = widgets.SelectionSlider(options=[2000, 5000, 20000, 50000, 200000], value=20000, description="λ")
g_apply = widgets.Button(description="Apply edit", button_style="success")
g_reset = widgets.Button(description="Restore original")
g_out = widgets.Output()

def g_run(*_):
    with g_out:
        clear_output(wait=True)
        subject = g_subject.value.strip()
        prompts = [p.strip() for p in g_prompts.value.split("\n") if p.strip()]
        tests = [t.strip() for t in g_tests.value.split("\n") if t.strip()]
        if not all(subject in p for p in prompts):
            print("Every edit prompt must contain the subject."); return
        gpt_restore()
        before = [gpt_generate(t) for t in tests]
        print(f"Editing layer {g_layer.value} with λ = {g_lam.value} (the first edit at a new layer also computes C0)...")
        log, size = gpt_edit(subject, prompts, " " + g_target.value.strip(), g_layer.value, g_lam.value)
        after = [gpt_generate(t) for t in tests]
        clear_output(wait=True)
        print(f"Target loss went from {log[0]:.2f} to {log[-1]:.2f}. Size of the change relative to the memory: {size:.4%}")
        display(pd.DataFrame({"Prompt": tests, "Before": before, "After": after}))

def g_restore(*_):
    gpt_restore()
    with g_out:
        clear_output(); print("Original GPT-2 XL restored.")

g_apply.on_click(g_run); g_reset.on_click(g_restore)
display(widgets.VBox([widgets.HBox([g_subject, g_target]), g_prompts, g_tests,
                      widgets.HBox([g_layer, g_lam]), widgets.HBox([g_apply, g_reset]), g_out]))

---
# Part 8. Your reflection

Write a reflection of **500 to 1,000 words** in the cell below. Before writing, look at the original work:

1. The MEMIT paper: Meng, Sen Sharma, Andonian, Belinkov, and Bau, *Mass-Editing Memory in a Transformer*, ICLR 2023. https://arxiv.org/abs/2210.07229. I also attached the paper to Week4 (Blackboard/Electronic Reserves)
2. The project page: https://memit.baulab.info
3. The official code: https://github.com/kmeng01/memit

Your reflection should cover three things, in your own words:

1. **Where could this be used?** Think of real situations where editing a model's facts would help, and situations where it could be risky or misused.
2. **Your experience running the homework.** What did you try in the labs, and what surprised you?
3. **What you learned.** What does this homework tell you about how a transformer stores and recalls facts, and about the limits of editing them?

You may also look back at any file in `memit_hw_data/` (rerun the file viewer in Part 1), for example `edit_log.csv`, which records every edit you made.

### My reflection

*Write your reflection here (double-click to edit).*

---
**Submission checklist**

1. Your GWID is entered in Part 1, and every cell has been run in order.
2. You completed all six stages of the Part 4 edit lab at least once.
3. Your reflection (500 to 1,000 words) is written in Part 8.
4. Save the notebook with its outputs and submit it.